In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

print("Temporal behavioral analysis started.")

Temporal behavioral analysis started.


In [2]:
fake_tweets_path = Path(
    "../data/raw/datasets_full.csv/fake_followers.csv/fake_followers.csv/tweets.csv"
)

print("Exists:", fake_tweets_path.exists())
print("Path:", fake_tweets_path)

Exists: True
Path: ..\data\raw\datasets_full.csv\fake_followers.csv\fake_followers.csv\tweets.csv


In [3]:
fake_tweets = pd.read_csv(
    fake_tweets_path,
    encoding="latin1"
)

print("Shape:", fake_tweets.shape)
print("Columns:")
print(fake_tweets.columns.tolist())

fake_tweets.head()

Shape: (196027, 23)
Columns:
['created_at', 'id', 'text', 'source', 'user_id', 'truncated', 'in_reply_to_status_id', 'in_reply_to_user_id', 'in_reply_to_screen_name', 'retweeted_status_id', 'geo', 'place', 'contributors', 'retweet_count', 'reply_count', 'favorite_count', 'favorited', 'retweeted', 'possibly_sensitive', 'num_hashtags', 'num_urls', 'num_mentions', 'timestamp']


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\2785169419.py:1: DtypeWarning: Columns (0: in_reply_to_screen_name, 1: place) have mixed types. Specify dtype option on import or set low_memory=False.
  fake_tweets = pd.read_csv(


,created_at,id,text,source,user_id,truncated,in_reply_to_status_id,in_reply_to_user_id,in_reply_to_screen_name,retweeted_status_id,...,retweet_count,reply_count,favorite_count,favorited,retweeted,possibly_sensitive,num_hashtags,num_urls,num_mentions,timestamp
0,Sat Apr 20 13:19:19 +0000 2013,325599560959393793,https://t.co/iocNIgHxXH. @LovesOfaLDNgirl her...,"<a href=""http://twitter.com/download/iphone"" r...",10935572,NaN,0,0,NaN,NaN,...,0,0,0,NaN,NaN,NaN,0,1,1,2013-04-20 15:19:19
1,Tue Apr 16 19:31:39 +0000 2013,324243711443730434,Well done hubby @Allan_76 http://t.co/AaeTwLucUG,"<a href=""http://instagram.com"" rel=""nofollow"">...",10935572,NaN,0,0,NaN,NaN,...,0,0,0,NaN,NaN,NaN,0,1,1,2013-04-16 21:31:39
2,Tue Apr 16 17:38:06 +0000 2013,324215137055670274,Two years with my lovely husband - thank you f...,"<a href=""http://instagram.com"" rel=""nofollow"">...",10935572,NaN,0,0,NaN,NaN,...,0,0,0,NaN,NaN,NaN,0,1,1,2013-04-16 19:38:06
3,Sun Apr 14 15:33:00 +0000 2013,323458877003792386,Sorry bunny about your ears but I was hungry.....,"<a href=""http://instagram.com"" rel=""nofollow"">...",10935572,NaN,0,0,NaN,NaN,...,0,0,0,NaN,NaN,NaN,0,1,0,2013-04-14 17:33:00
4,Fri Apr 12 15:37:59 +0000 2013,322735354148945920,"Small man, big drink @Allan_76 http://t.co/4NU...","<a href=""http://instagram.com"" rel=""nofollow"">...",10935572,NaN,0,0,NaN,NaN,...,0,1,0,NaN,NaN,NaN,0,1,1,2013-04-12 17:37:59


In [4]:
# Keep only the columns we need
temp = fake_tweets[["user_id", "created_at"]].copy()

# Convert timestamps
temp["created_at"] = pd.to_datetime(
    temp["created_at"],
    errors="coerce"
)

# Remove invalid timestamps
temp = temp.dropna(subset=["user_id", "created_at"])

# Sort by account and time
temp = temp.sort_values(["user_id", "created_at"])

# Calculate time difference between consecutive tweets
temp["intertweet_seconds"] = (
    temp.groupby("user_id")["created_at"]
        .diff()
        .dt.total_seconds()
)

# Calculate mean interval for each account
fake_mean_intertweet = (
    temp.groupby("user_id")["intertweet_seconds"]
        .mean()
        .reset_index(name="mean_intertweet_seconds")
)

print("Accounts:", len(fake_mean_intertweet))
fake_mean_intertweet.head()

C:\Users\namit\AppData\Local\Temp\ipykernel_1832\998346039.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp["created_at"] = pd.to_datetime(


Accounts: 3202


,user_id,mean_intertweet_seconds
0,10935572,3.079435e+05
1,16119337,4.881397e+04
2,16753788,4.717938e+06
3,17640121,6.130340e+06
4,17656600,4.257927e+05


In [5]:
fake_mean_intertweet["mean_intertweet_seconds"].describe()

count    3.088000e+03
mean     1.256946e+06
std      4.625927e+06
min      2.000000e+00
25%      3.341127e+05
50%      4.725721e+05
75%      8.545844e+05
max      1.036911e+08
Name: mean_intertweet_seconds, dtype: float64

In [6]:
print("Number of accounts:", len(fake_mean_intertweet))

print("\nMissing values:")
print(fake_mean_intertweet["mean_intertweet_seconds"].isna().sum())

print("\nDescriptive statistics:")
print(fake_mean_intertweet["mean_intertweet_seconds"].describe())

print("\nLargest 10 mean intervals:")
print(
    fake_mean_intertweet
    .sort_values("mean_intertweet_seconds", ascending=False)
    .head(10)
)

Number of accounts: 3202

Missing values:
114

Descriptive statistics:
count    3.088000e+03
mean     1.256946e+06
std      4.625927e+06
min      2.000000e+00
25%      3.341127e+05
50%      4.725721e+05
75%      8.545844e+05
max      1.036911e+08
Name: mean_intertweet_seconds, dtype: float64

Largest 10 mean intervals:
       user_id  mean_intertweet_seconds
57    56163699              103691143.0
194  119926900               97241502.0
330  190265169               80967303.0
147   96871607               72371346.0
244  142150030               64757158.0
441  254839933               59504813.0
531  328246965               57596892.0
120   82122078               49204855.5
286  163142123               44306359.0
502  303469076               36854702.0


In [7]:
fake_mean_intertweet["mean_intertweet_hours"] = (
    fake_mean_intertweet["mean_intertweet_seconds"] / 3600
)

print(
    fake_mean_intertweet["mean_intertweet_hours"].describe()
)

count     3088.000000
mean       349.151675
std       1284.979825
min          0.000556
25%         92.809093
50%        131.270016
75%        237.384552
max      28803.095278
Name: mean_intertweet_hours, dtype: float64


In [8]:
tweet_counts = (
    temp.groupby("user_id")
    .size()
    .reset_index(name="tweet_count")
)

fake_mean_intertweet = fake_mean_intertweet.merge(
    tweet_counts,
    on="user_id",
    how="left"
)

print(fake_mean_intertweet["tweet_count"].describe())

count    3202.000000
mean       61.220175
std       232.266613
min         1.000000
25%        17.000000
50%        23.000000
75%        40.000000
max      3383.000000
Name: tweet_count, dtype: float64


In [9]:
print(
    fake_mean_intertweet[
        ["user_id", "tweet_count", "mean_intertweet_seconds"]
    ].head(10)
)

    user_id  tweet_count  mean_intertweet_seconds
0  10935572          556             3.079435e+05
1  16119337          355             4.881397e+04
2  16753788           30             4.717938e+06
3  17640121           20             6.130340e+06
4  17656600          306             4.257927e+05
5  19230427           96             6.695184e+05
6  19951698          193             1.416404e+05
7  20731719          436             3.071765e+05
8  21196037           78             1.568908e+06
9  21375100          218             5.812848e+05


In [10]:
median_intertweet = (
    temp.groupby("user_id")["intertweet_seconds"]
        .median()
        .reset_index(name="median_intertweet_seconds")
)

print("Accounts:", len(median_intertweet))

print("\nStatistics:")
print(median_intertweet["median_intertweet_seconds"].describe())

print("\nFirst 10 accounts:")
print(median_intertweet.head(10))

Accounts: 3202

Statistics:
count    3.088000e+03
mean     6.974866e+05
std      4.272561e+06
min      2.000000e+00
25%      1.762214e+05
50%      2.694390e+05
75%      3.807890e+05
max      1.036911e+08
Name: median_intertweet_seconds, dtype: float64

First 10 accounts:
    user_id  median_intertweet_seconds
0  10935572                    24136.0
1  16119337                       55.5
2  16753788                  1798473.0
3  17640121                  1517379.0
4  17656600                    35335.0
5  19230427                   172683.0
6  19951698                       20.0
7  20731719                       11.0
8  21196037                     7696.0
9  21375100                    69498.0


In [11]:
median_intertweet["median_intertweet_hours"] = (
    median_intertweet["median_intertweet_seconds"] / 3600
)

print(median_intertweet["median_intertweet_hours"].describe())

count     3088.000000
mean       193.746291
std       1186.822497
min          0.000556
25%         48.950382
50%         74.844167
75%        105.774722
max      28803.095278
Name: median_intertweet_hours, dtype: float64


In [12]:
temp["intertweet_seconds"]

553              NaN
552       36893348.0
551           5911.0
550         192524.0
549          85088.0
             ...    
196022           NaN
196024           NaN
196023     1163404.0
196026           NaN
196025     1164225.0
Name: intertweet_seconds, Length: 196027, dtype: float64

In [13]:
intertweet_std = (
    temp.groupby("user_id")["intertweet_seconds"]
        .std()
        .reset_index(name="intertweet_std_seconds")
)

print("Accounts:", len(intertweet_std))

print("\nStatistics:")
print(intertweet_std["intertweet_std_seconds"].describe())

Accounts: 3202

Statistics:
count    2.998000e+03
mean     1.754492e+06
std      4.639735e+06
min      0.000000e+00
25%      3.123197e+05
50%      6.436917e+05
75%      1.233665e+06
max      6.952952e+07
Name: intertweet_std_seconds, dtype: float64


In [14]:
intertweet_stats = (
    temp.groupby("user_id")["intertweet_seconds"]
        .agg(
            mean_intertweet_seconds="mean",
            std_intertweet_seconds="std"
        )
        .reset_index()
)

intertweet_stats["intertweet_cv"] = (
    intertweet_stats["std_intertweet_seconds"] /
    intertweet_stats["mean_intertweet_seconds"]
)

print(intertweet_stats.head(10))

print("\nCV statistics:")
print(intertweet_stats["intertweet_cv"].describe())

    user_id  mean_intertweet_seconds  std_intertweet_seconds  intertweet_cv
0  10935572             3.079435e+05            2.074548e+06       6.736780
1  16119337             4.881397e+04            5.457522e+05      11.180248
2  16753788             4.717938e+06            5.885562e+06       1.247486
3  17640121             6.130340e+06            8.649692e+06       1.410965
4  17656600             4.257927e+05            1.317960e+06       3.095309
5  19230427             6.695184e+05            1.408532e+06       2.103799
6  19951698             1.416404e+05            7.618125e+05       5.378496
7  20731719             3.071765e+05            4.407376e+06      14.348022
8  21196037             1.568908e+06            6.758710e+06       4.307907
9  21375100             5.812848e+05            2.142826e+06       3.686362

CV statistics:
count    2998.000000
mean        1.640041
std         1.829862
min         0.000000
25%         0.905957
50%         1.227131
75%         1.577075
m

In [15]:
intertweet_stats["burstiness"] = (
    intertweet_stats["std_intertweet_seconds"]
    - intertweet_stats["mean_intertweet_seconds"]
) / (
    intertweet_stats["std_intertweet_seconds"]
    + intertweet_stats["mean_intertweet_seconds"]
)

In [16]:
print("Burstiness statistics:")
print(intertweet_stats["burstiness"].describe())

Burstiness statistics:
count    2998.000000
mean        0.117217
std         0.242291
min        -1.000000
25%        -0.049342
50%         0.101984
75%         0.223926
max         0.942490
Name: burstiness, dtype: float64


In [17]:
print("\nBurstiness range:")
print(
    intertweet_stats["burstiness"].min(),
    "to",
    intertweet_stats["burstiness"].max()
)


Burstiness range:
-1.0 to 0.9424900012270045


In [18]:
print("\nFirst 10 accounts:")
print(
    intertweet_stats[
        ["user_id",
         "mean_intertweet_seconds",
         "std_intertweet_seconds",
         "intertweet_cv",
         "burstiness"]
    ].head(10)
)


First 10 accounts:
    user_id  mean_intertweet_seconds  std_intertweet_seconds  intertweet_cv  \
0  10935572             3.079435e+05            2.074548e+06       6.736780   
1  16119337             4.881397e+04            5.457522e+05      11.180248   
2  16753788             4.717938e+06            5.885562e+06       1.247486   
3  17640121             6.130340e+06            8.649692e+06       1.410965   
4  17656600             4.257927e+05            1.317960e+06       3.095309   
5  19230427             6.695184e+05            1.408532e+06       2.103799   
6  19951698             1.416404e+05            7.618125e+05       5.378496   
7  20731719             3.071765e+05            4.407376e+06      14.348022   
8  21196037             1.568908e+06            6.758710e+06       4.307907   
9  21375100             5.812848e+05            2.142826e+06       3.686362   

   burstiness  
0    0.741495  
1    0.835800  
2    0.110117  
3    0.170456  
4    0.511636  
5    0.355628 

In [19]:
temp["posting_hour"] = temp["created_at"].dt.hour

print(temp[["user_id", "created_at", "posting_hour"]].head(10))

      user_id                created_at  posting_hour
553  10935572 2007-12-07 13:57:21+00:00            13
552  10935572 2009-02-06 14:06:29+00:00            14
551  10935572 2009-02-06 15:45:00+00:00            15
550  10935572 2009-02-08 21:13:44+00:00            21
549  10935572 2009-02-09 20:51:52+00:00            20
548  10935572 2009-02-14 19:12:33+00:00            19
547  10935572 2009-02-15 21:31:16+00:00            21
546  10935572 2009-02-28 20:30:01+00:00            20
545  10935572 2009-06-26 16:20:09+00:00            16
544  10935572 2009-08-12 17:23:59+00:00            17


In [20]:
def calculate_hour_entropy(hours):
    counts = hours.value_counts(normalize=True)
    return -(counts * np.log2(counts)).sum()

hour_entropy = (
    temp.groupby("user_id")["posting_hour"]
        .apply(calculate_hour_entropy)
        .reset_index(name="posting_time_entropy")
)

print("Accounts:", len(hour_entropy))
print("\nEntropy statistics:")
print(hour_entropy["posting_time_entropy"].describe())

Accounts: 3202

Entropy statistics:
count    3202.000000
mean        3.342091
std         1.023852
min        -0.000000
25%         3.251629
50%         3.628785
75%         3.960362
max         4.514577
Name: posting_time_entropy, dtype: float64


In [21]:
print(
    "Entropy range:",
    hour_entropy["posting_time_entropy"].min(),
    "to",
    hour_entropy["posting_time_entropy"].max()
)

Entropy range: -0.0 to 4.514577483453128


In [22]:
active_hour_count = (
    temp.groupby("user_id")["posting_hour"]
        .nunique()
        .reset_index(name="active_hour_count")
)

print("Accounts:", len(active_hour_count))

print("\nStatistics:")
print(active_hour_count["active_hour_count"].describe())

Accounts: 3202

Statistics:
count    3202.000000
mean       13.791380
std         5.964716
min         1.000000
25%        11.000000
50%        14.000000
75%        18.000000
max        24.000000
Name: active_hour_count, dtype: float64


In [23]:
print("\nValue counts:")
print(
    active_hour_count["active_hour_count"]
    .value_counts()
    .sort_index()
)


Value counts:
active_hour_count
1     131
2     117
3      73
4      47
5      36
6      30
7      36
8      48
9      84
10    119
11    199
12    264
13    288
14    242
15    244
16    188
17    162
18    137
19    131
20    144
21    154
22    150
23    106
24     72
Name: count, dtype: int64


In [24]:
fake_temporal_features = (
    fake_mean_intertweet
    .merge(
        intertweet_stats[
            ["user_id", "intertweet_cv", "burstiness"]
        ],
        on="user_id",
        how="left"
    )
    .merge(
        median_intertweet[
            ["user_id", "median_intertweet_seconds"]
        ],
        on="user_id",
        how="left"
    )
    .merge(
        hour_entropy,
        on="user_id",
        how="left"
    )
    .merge(
        active_hour_count,
        on="user_id",
        how="left"
    )
)

print("Shape:", fake_temporal_features.shape)
print("\nColumns:")
print(fake_temporal_features.columns.tolist())

fake_temporal_features.head()

Shape: (3202, 9)

Columns:
['user_id', 'mean_intertweet_seconds', 'mean_intertweet_hours', 'tweet_count', 'intertweet_cv', 'burstiness', 'median_intertweet_seconds', 'posting_time_entropy', 'active_hour_count']


,user_id,mean_intertweet_seconds,mean_intertweet_hours,tweet_count,intertweet_cv,burstiness,median_intertweet_seconds,posting_time_entropy,active_hour_count
0,10935572,3.079435e+05,85.539871,556,6.736780,0.741495,24136.0,4.051593,21
1,16119337,4.881397e+04,13.559435,355,11.180248,0.835800,55.5,3.547240,19
2,16753788,4.717938e+06,1310.538199,30,1.247486,0.110117,1798473.0,3.694740,15
3,17640121,6.130340e+06,1702.872266,20,1.410965,0.170456,1517379.0,3.246439,11
4,17656600,4.257927e+05,118.275749,306,3.095309,0.511636,35335.0,4.169805,24


In [25]:
temp["is_night"] = temp["posting_hour"].between(0, 5)

night_activity_ratio = (
    temp.groupby("user_id")["is_night"]
        .mean()
        .reset_index(name="night_activity_ratio")
)

print("Accounts:", len(night_activity_ratio))

print("\nStatistics:")
print(night_activity_ratio["night_activity_ratio"].describe())

Accounts: 3202

Statistics:
count    3202.000000
mean        0.215119
std         0.162782
min         0.000000
25%         0.125000
50%         0.193265
75%         0.268657
max         1.000000
Name: night_activity_ratio, dtype: float64


In [26]:
print(
    "Range:",
    night_activity_ratio["night_activity_ratio"].min(),
    "to",
    night_activity_ratio["night_activity_ratio"].max()
)

print("\nFirst 10:")
print(night_activity_ratio.head(10))

Range: 0.0 to 1.0

First 10:
    user_id  night_activity_ratio
0  10935572              0.026978
1  16119337              0.473239
2  16753788              0.266667
3  17640121              0.400000
4  17656600              0.333333
5  19230427              0.010417
6  19951698              0.046632
7  20731719              0.236239
8  21196037              0.217949
9  21375100              0.495413


In [27]:
fake_temporal_features = fake_temporal_features.merge(
    night_activity_ratio,
    on="user_id",
    how="left"
)

print("Shape:", fake_temporal_features.shape)
print(fake_temporal_features.columns.tolist())

Shape: (3202, 10)
['user_id', 'mean_intertweet_seconds', 'mean_intertweet_hours', 'tweet_count', 'intertweet_cv', 'burstiness', 'median_intertweet_seconds', 'posting_time_entropy', 'active_hour_count', 'night_activity_ratio']


In [28]:
temp["weekday"] = temp["created_at"].dt.weekday

temp["is_weekend"] = temp["weekday"].isin([5, 6])

weekend_activity_ratio = (
    temp.groupby("user_id")["is_weekend"]
        .mean()
        .reset_index(name="weekend_activity_ratio")
)

print("Accounts:", len(weekend_activity_ratio))

print("\nStatistics:")
print(weekend_activity_ratio["weekend_activity_ratio"].describe())

Accounts: 3202

Statistics:
count    3202.000000
mean        0.264214
std         0.145217
min         0.000000
25%         0.191255
50%         0.263158
75%         0.333333
max         1.000000
Name: weekend_activity_ratio, dtype: float64


In [29]:
print(
    "Range:",
    weekend_activity_ratio["weekend_activity_ratio"].min(),
    "to",
    weekend_activity_ratio["weekend_activity_ratio"].max()
)

print("\nFirst 10:")
print(weekend_activity_ratio.head(10))

Range: 0.0 to 1.0

First 10:
    user_id  weekend_activity_ratio
0  10935572                0.320144
1  16119337                0.016901
2  16753788                0.166667
3  17640121                0.250000
4  17656600                0.199346
5  19230427                0.312500
6  19951698                0.626943
7  20731719                0.332569
8  21196037                0.282051
9  21375100                0.266055


In [30]:
fake_temporal_features = fake_temporal_features.merge(
    weekend_activity_ratio,
    on="user_id",
    how="left"
)

print("Shape:", fake_temporal_features.shape)
print(fake_temporal_features.columns.tolist())

Shape: (3202, 11)
['user_id', 'mean_intertweet_seconds', 'mean_intertweet_hours', 'tweet_count', 'intertweet_cv', 'burstiness', 'median_intertweet_seconds', 'posting_time_entropy', 'active_hour_count', 'night_activity_ratio', 'weekend_activity_ratio']


In [31]:
temp["date"] = temp["created_at"].dt.date

daily_counts = (
    temp.groupby(["user_id", "date"])
        .size()
        .reset_index(name="daily_tweet_count")
)

print("Account-day records:", len(daily_counts))

print("\nDaily tweet count statistics:")
print(daily_counts["daily_tweet_count"].describe())

Account-day records: 87942

Daily tweet count statistics:
count    87942.000000
mean         2.229049
std          8.195446
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max        579.000000
Name: daily_tweet_count, dtype: float64


In [32]:
daily_activity_stats = (
    daily_counts.groupby("user_id")["daily_tweet_count"]
        .agg(
            mean_daily_tweets="mean",
            std_daily_tweets="std"
        )
        .reset_index()
)

daily_activity_stats["daily_activity_cv"] = (
    daily_activity_stats["std_daily_tweets"] /
    daily_activity_stats["mean_daily_tweets"]
)

print("Accounts:", len(daily_activity_stats))

print("\nDaily activity CV statistics:")
print(daily_activity_stats["daily_activity_cv"].describe())

Accounts: 3202

Daily activity CV statistics:
count    3061.000000
mean        0.396657
std         0.377953
min         0.000000
25%         0.235294
50%         0.317744
75%         0.450533
max         4.555788
Name: daily_activity_cv, dtype: float64


In [33]:
print(
    daily_activity_stats[
        ["user_id", "mean_daily_tweets",
         "std_daily_tweets", "daily_activity_cv"]
    ].head(10)
)

    user_id  mean_daily_tweets  std_daily_tweets  daily_activity_cv
0  10935572           2.206349          1.890709           0.856940
1  16119337          39.444444         76.313680           1.934713
2  16753788           1.304348          1.258960           0.965203
3  17640121           1.176471          0.392953           0.334010
4  17656600           1.987013          1.652833           0.831818
5  19230427           1.411765          0.737792           0.522603
6  19951698           6.031250         14.894217           2.469507
7  20731719          24.222222         21.821618           0.900892
8  21196037           2.516129          2.079599           0.826507
9  21375100           1.772358          1.304551           0.736054


In [34]:
top_hour_concentration = (
    temp.groupby("user_id")["posting_hour"]
        .value_counts(normalize=True)
        .groupby(level=0)
        .max()
        .reset_index(name="top_hour_concentration")
)

print("Accounts:", len(top_hour_concentration))

print("\nStatistics:")
print(top_hour_concentration["top_hour_concentration"].describe())

Accounts: 3202

Statistics:
count    3202.000000
mean        0.217443
std         0.194591
min         0.067086
25%         0.121212
50%         0.153846
75%         0.210526
max         1.000000
Name: top_hour_concentration, dtype: float64


In [35]:
print(
    "Range:",
    top_hour_concentration["top_hour_concentration"].min(),
    "to",
    top_hour_concentration["top_hour_concentration"].max()
)

print("\nFirst 10:")
print(top_hour_concentration.head(10))

Range: 0.06708595387840671 to 1.0

First 10:
    user_id  top_hour_concentration
0  10935572                0.129496
1  16119337                0.191549
2  16753788                0.166667
3  17640121                0.200000
4  17656600                0.104575
5  19230427                0.156250
6  19951698                0.466321
7  20731719                0.126147
8  21196037                0.205128
9  21375100                0.155963


In [36]:
fake_temporal_features = fake_temporal_features.merge(
    top_hour_concentration,
    on="user_id",
    how="left"
)

print("Shape:", fake_temporal_features.shape)
print(fake_temporal_features.columns.tolist())

Shape: (3202, 12)
['user_id', 'mean_intertweet_seconds', 'mean_intertweet_hours', 'tweet_count', 'intertweet_cv', 'burstiness', 'median_intertweet_seconds', 'posting_time_entropy', 'active_hour_count', 'night_activity_ratio', 'weekend_activity_ratio', 'top_hour_concentration']


In [37]:
import pandas as pd
import numpy as np
from pathlib import Path

In [58]:
def build_temporal_features(tweets_df):

    # ---------------------------------------------------------
    # 1. Select required columns
    # ---------------------------------------------------------
    temp = tweets_df[["user_id", "created_at"]].copy()

    # ---------------------------------------------------------
    # 2. Clean timestamp values
    # ---------------------------------------------------------
    created_raw = temp["created_at"].astype(str).str.strip()

    # Remove trailing L from Unix timestamps
    created_clean = created_raw.str.replace(
        r"L$",
        "",
        regex=True
    )

    # ---------------------------------------------------------
    # 3. Create empty UTC datetime Series
    # ---------------------------------------------------------
    parsed_dates = pd.Series(
        pd.NaT,
        index=temp.index,
        dtype="datetime64[ns, UTC]"
    )

    # ---------------------------------------------------------
    # 4. Detect Unix millisecond timestamps
    # ---------------------------------------------------------
    is_unix_ms = created_clean.str.fullmatch(
        r"\d{10,15}"
    )

    # Parse Unix millisecond timestamps
    if is_unix_ms.any():

        parsed_dates.loc[is_unix_ms] = pd.to_datetime(
            pd.to_numeric(
                created_clean.loc[is_unix_ms],
                errors="coerce"
            ),
            unit="ms",
            errors="coerce",
            utc=True
        )

    # ---------------------------------------------------------
    # 5. Parse normal datetime strings
    # ---------------------------------------------------------
    normal_mask = ~is_unix_ms

    if normal_mask.any():

        parsed_dates.loc[normal_mask] = pd.to_datetime(
            created_clean.loc[normal_mask],
            errors="coerce",
            utc=True
        )

    # ---------------------------------------------------------
    # 6. Convert timezone-aware timestamps to naive UTC
    # ---------------------------------------------------------
    parsed_dates = parsed_dates.dt.tz_localize(None)

    temp["created_at"] = parsed_dates

    # ---------------------------------------------------------
    # 7. Remove invalid timestamps
    # ---------------------------------------------------------
    temp = temp.dropna(
        subset=["user_id", "created_at"]
    )

    # ---------------------------------------------------------
    # 8. Sort tweets chronologically for each account
    # ---------------------------------------------------------
    temp = temp.sort_values(
        ["user_id", "created_at"]
    )

    # ---------------------------------------------------------
    # 9. Calculate inter-tweet time
    # ---------------------------------------------------------
    temp["intertweet_seconds"] = (
        temp.groupby("user_id")["created_at"]
        .diff()
        .dt.total_seconds()
    )

    # ---------------------------------------------------------
    # 10. Inter-tweet statistics
    # ---------------------------------------------------------
    intertweet_stats = (
        temp.groupby("user_id")["intertweet_seconds"]
        .agg(
            mean_intertweet_seconds="mean",
            median_intertweet_seconds="median",
            std_intertweet_seconds="std"
        )
        .reset_index()
    )

    # ---------------------------------------------------------
    # 11. Inter-tweet coefficient of variation
    # ---------------------------------------------------------
    intertweet_stats["intertweet_cv"] = (
        intertweet_stats["std_intertweet_seconds"]
        /
        intertweet_stats["mean_intertweet_seconds"]
    )

    # ---------------------------------------------------------
    # 12. Burstiness
    # ---------------------------------------------------------
    intertweet_stats["burstiness"] = (
        (
            intertweet_stats["std_intertweet_seconds"]
            -
            intertweet_stats["mean_intertweet_seconds"]
        )
        /
        (
            intertweet_stats["std_intertweet_seconds"]
            +
            intertweet_stats["mean_intertweet_seconds"]
        )
    )

    # ---------------------------------------------------------
    # 13. Posting hour
    # ---------------------------------------------------------
    temp["posting_hour"] = (
        temp["created_at"].dt.hour
    )

    # ---------------------------------------------------------
    # 14. Posting-time entropy
    # ---------------------------------------------------------
    def calculate_hour_entropy(hours):

        counts = hours.value_counts(
            normalize=True
        )

        return -(
            counts * np.log2(counts)
        ).sum()

    hour_entropy = (
        temp.groupby("user_id")["posting_hour"]
        .apply(calculate_hour_entropy)
        .reset_index(
            name="posting_time_entropy"
        )
    )

    # ---------------------------------------------------------
    # 15. Number of active hours
    # ---------------------------------------------------------
    active_hour_count = (
        temp.groupby("user_id")["posting_hour"]
        .nunique()
        .reset_index(
            name="active_hour_count"
        )
    )

    # ---------------------------------------------------------
    # 16. Night activity ratio
    # 00:00 - 05:59
    # ---------------------------------------------------------
    temp["is_night"] = (
        temp["posting_hour"].between(0, 5)
    )

    night_activity_ratio = (
        temp.groupby("user_id")["is_night"]
        .mean()
        .reset_index(
            name="night_activity_ratio"
        )
    )

    # ---------------------------------------------------------
    # 17. Weekend activity ratio
    # Saturday = 5
    # Sunday = 6
    # ---------------------------------------------------------
    temp["weekday"] = (
        temp["created_at"].dt.weekday
    )

    temp["is_weekend"] = (
        temp["weekday"].isin([5, 6])
    )

    weekend_activity_ratio = (
        temp.groupby("user_id")["is_weekend"]
        .mean()
        .reset_index(
            name="weekend_activity_ratio"
        )
    )

    # ---------------------------------------------------------
    # 18. Top-hour concentration
    # ---------------------------------------------------------
    top_hour_concentration = (
        temp.groupby("user_id")["posting_hour"]
        .value_counts(normalize=True)
        .groupby(level=0)
        .max()
        .reset_index(
            name="top_hour_concentration"
        )
    )

    # ---------------------------------------------------------
    # 19. Daily tweet activity
    # ---------------------------------------------------------
    temp["date"] = (
        temp["created_at"].dt.date
    )

    daily_counts = (
        temp.groupby(
            ["user_id", "date"]
        )
        .size()
        .reset_index(
            name="daily_tweet_count"
        )
    )

    daily_activity_stats = (
        daily_counts.groupby("user_id")[
            "daily_tweet_count"
        ]
        .agg(
            mean_daily_tweets="mean",
            std_daily_tweets="std"
        )
        .reset_index()
    )

    # ---------------------------------------------------------
    # 20. Daily activity coefficient of variation
    # ---------------------------------------------------------
    daily_activity_stats["daily_activity_cv"] = (
        daily_activity_stats["std_daily_tweets"]
        /
        daily_activity_stats["mean_daily_tweets"]
    )

    # ---------------------------------------------------------
    # 21. Tweet count used
    # ---------------------------------------------------------
    tweet_count = (
        temp.groupby("user_id")
        .size()
        .reset_index(
            name="tweet_count"
        )
    )

    # ---------------------------------------------------------
    # 22. Build final temporal feature table
    # ---------------------------------------------------------
    temporal_features = intertweet_stats[
        [
            "user_id",
            "mean_intertweet_seconds",
            "median_intertweet_seconds",
            "intertweet_cv",
            "burstiness"
        ]
    ].copy()

    temporal_features = temporal_features.merge(
        hour_entropy,
        on="user_id",
        how="left"
    )

    temporal_features = temporal_features.merge(
        active_hour_count,
        on="user_id",
        how="left"
    )

    temporal_features = temporal_features.merge(
        night_activity_ratio,
        on="user_id",
        how="left"
    )

    temporal_features = temporal_features.merge(
        weekend_activity_ratio,
        on="user_id",
        how="left"
    )

    temporal_features = temporal_features.merge(
        top_hour_concentration,
        on="user_id",
        how="left"
    )

    temporal_features = temporal_features.merge(
        daily_activity_stats[
            ["user_id", "daily_activity_cv"]
        ],
        on="user_id",
        how="left"
    )

    temporal_features = temporal_features.merge(
        tweet_count,
        on="user_id",
        how="left"
    )

    return temporal_features

In [39]:
dataset_paths = {
    "fake_followers":
        "../data/raw/datasets_full.csv/fake_followers.csv/fake_followers.csv/tweets.csv",

    "genuine_accounts":
        "../data/raw/datasets_full.csv/genuine_accounts.csv/genuine_accounts.csv/tweets.csv",

    "social_spambots_1":
        "../data/raw/datasets_full.csv/social_spambots_1.csv/social_spambots_1.csv/tweets.csv",

    "social_spambots_2":
        "../data/raw/datasets_full.csv/social_spambots_2.csv/social_spambots_2.csv/tweets.csv",

    "social_spambots_3":
        "../data/raw/datasets_full.csv/social_spambots_3.csv/social_spambots_3.csv/tweets.csv",

    "traditional_spambots_1":
        "../data/raw/datasets_full.csv/traditional_spambots_1.csv/traditional_spambots_1.csv/tweets.csv"
}

print("Datasets:", len(dataset_paths))

Datasets: 6


In [41]:
from pathlib import Path

base_path = Path("../data/raw/datasets_full.csv")

matches = list(base_path.rglob("genuine_accounts*"))

for p in matches:
    print(p)

..\data\raw\datasets_full.csv\genuine_accounts.csv


In [42]:
from pathlib import Path

genuine_path = Path("../data/raw/datasets_full.csv/genuine_accounts.csv")

print("Exists:", genuine_path.exists())
print("\nContents:")

for p in genuine_path.iterdir():
    print(p)

Exists: True

Contents:
..\data\raw\datasets_full.csv\genuine_accounts.csv\tweets.csv
..\data\raw\datasets_full.csv\genuine_accounts.csv\users.csv


In [43]:
from pathlib import Path

for account_type, path in dataset_paths.items():
    p = Path(path)
    print(
        f"{account_type:25} | "
        f"{'FOUND' if p.exists() else 'MISSING'} | "
        f"{p}"
    )

fake_followers            | FOUND | ..\data\raw\datasets_full.csv\fake_followers.csv\fake_followers.csv\tweets.csv
genuine_accounts          | MISSING | ..\data\raw\datasets_full.csv\genuine_accounts.csv\genuine_accounts.csv\tweets.csv
social_spambots_1         | FOUND | ..\data\raw\datasets_full.csv\social_spambots_1.csv\social_spambots_1.csv\tweets.csv
social_spambots_2         | FOUND | ..\data\raw\datasets_full.csv\social_spambots_2.csv\social_spambots_2.csv\tweets.csv
social_spambots_3         | FOUND | ..\data\raw\datasets_full.csv\social_spambots_3.csv\social_spambots_3.csv\tweets.csv
traditional_spambots_1    | FOUND | ..\data\raw\datasets_full.csv\traditional_spambots_1.csv\traditional_spambots_1.csv\tweets.csv


In [44]:
dataset_paths = {
    "fake_followers":
        "../data/raw/datasets_full.csv/fake_followers.csv/fake_followers.csv/tweets.csv",

    "genuine_accounts":
        "../data/raw/datasets_full.csv/genuine_accounts.csv/tweets.csv",

    "social_spambots_1":
        "../data/raw/datasets_full.csv/social_spambots_1.csv/social_spambots_1.csv/tweets.csv",

    "social_spambots_2":
        "../data/raw/datasets_full.csv/social_spambots_2.csv/social_spambots_2.csv/tweets.csv",

    "social_spambots_3":
        "../data/raw/datasets_full.csv/social_spambots_3.csv/social_spambots_3.csv/tweets.csv",

    "traditional_spambots_1":
        "../data/raw/datasets_full.csv/traditional_spambots_1.csv/traditional_spambots_1.csv/tweets.csv"
}

In [45]:
for account_type, path in dataset_paths.items():
    p = Path(path)
    print(
        f"{account_type:25} | "
        f"{'FOUND' if p.exists() else 'MISSING'} | "
        f"{p}"
    )

fake_followers            | FOUND | ..\data\raw\datasets_full.csv\fake_followers.csv\fake_followers.csv\tweets.csv
genuine_accounts          | FOUND | ..\data\raw\datasets_full.csv\genuine_accounts.csv\tweets.csv
social_spambots_1         | FOUND | ..\data\raw\datasets_full.csv\social_spambots_1.csv\social_spambots_1.csv\tweets.csv
social_spambots_2         | FOUND | ..\data\raw\datasets_full.csv\social_spambots_2.csv\social_spambots_2.csv\tweets.csv
social_spambots_3         | FOUND | ..\data\raw\datasets_full.csv\social_spambots_3.csv\social_spambots_3.csv\tweets.csv
traditional_spambots_1    | FOUND | ..\data\raw\datasets_full.csv\traditional_spambots_1.csv\traditional_spambots_1.csv\tweets.csv


In [46]:
all_temporal_features = {}

for account_type, path in dataset_paths.items():
    print("\n" + "=" * 60)
    print("Processing:", account_type)

    tweets = pd.read_csv(
        path,
        encoding="latin1"
    )

    print("Tweets:", len(tweets))

    features = build_temporal_features(tweets)
    features["account_type"] = account_type

    all_temporal_features[account_type] = features

    print("Accounts:", len(features))
    print("Feature shape:", features.shape)

    del tweets


Processing: fake_followers


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\1138504534.py:7: DtypeWarning: Columns (0: in_reply_to_screen_name, 1: place) have mixed types. Specify dtype option on import or set low_memory=False.
  tweets = pd.read_csv(
C:\Users\namit\AppData\Local\Temp\ipykernel_1832\831449219.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp["created_at"] = pd.to_datetime(


Tweets: 196027
Accounts: 3202
Feature shape: (3202, 13)

Processing: genuine_accounts


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\1138504534.py:7: DtypeWarning: Columns (0: id) have mixed types. Specify dtype option on import or set low_memory=False.
  tweets = pd.read_csv(


Tweets: 2839362


...

Tweets: 1418557


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\831449219.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp["created_at"] = pd.to_datetime(


Accounts: 464
Feature shape: (464, 13)

Processing: traditional_spambots_1
Tweets: 145094


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\831449219.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp["created_at"] = pd.to_datetime(


Accounts: 0
Feature shape: (0, 13)


In [47]:
temporal_dataset = pd.concat(
    all_temporal_features.values(),
    ignore_index=True
)

print("Final shape:", temporal_dataset.shape)

print("\nAccounts by category:")
print(
    temporal_dataset["account_type"]
    .value_counts()
)

Final shape: (9197, 13)

Accounts by category:
account_type
social_spambots_2    3457
fake_followers       3202
genuine_accounts     1083
social_spambots_1     991
social_spambots_3     464
Name: count, dtype: int64


In [48]:
temporal_dataset["label"] = (
    temporal_dataset["account_type"]
    != "genuine_accounts"
).astype(int)

In [49]:
print(
    temporal_dataset["label"]
    .value_counts()
)

label
1    8114
0    1083
Name: count, dtype: int64


In [50]:
print("Duplicate user IDs:",
      temporal_dataset["user_id"].duplicated().sum())

print("\nMissing values:")
print(
    temporal_dataset.isna().sum()
)

print("\nShape:")
print(temporal_dataset.shape)

Duplicate user IDs: 0

Missing values:
user_id                        0
mean_intertweet_seconds      114
median_intertweet_seconds    114
intertweet_cv                204
burstiness                   204
posting_time_entropy           0
active_hour_count              0
night_activity_ratio           0
weekend_activity_ratio         0
top_hour_concentration         0
daily_activity_cv            142
tweet_count                    0
account_type                   0
label                          0
dtype: int64

Shape:
(9197, 14)


In [51]:
trad_path = dataset_paths["traditional_spambots_1"]

trad_tweets = pd.read_csv(
    trad_path,
    encoding="latin1"
)

print("Shape:", trad_tweets.shape)
print("\ncreated_at sample:")
print(trad_tweets["created_at"].head(10))

print("\ncreated_at non-null:")
print(trad_tweets["created_at"].notna().sum())

print("\ncreated_at dtype:")
print(trad_tweets["created_at"].dtype)

Shape: (145094, 25)

created_at sample:
0    1283282654000L
1    1283282651000L
2    1283282592000L
3    1283282571000L
4    1283282543000L
5    1283282523000L
6    1283282509000L
7    1283282494000L
8    1282636240000L
9    1281125480000L
Name: created_at, dtype: str

created_at non-null:
145094

created_at dtype:
str


In [52]:
print(trad_tweets[["user_id", "created_at"]].head(20).to_string())

    user_id      created_at
0   7248952  1283282654000L
1   7248952  1283282651000L
2   7248952  1283282592000L
3   7248952  1283282571000L
4   7248952  1283282543000L
5   7248952  1283282523000L
6   7248952  1283282509000L
7   7248952  1283282494000L
8   7248952  1282636240000L
9   7248952  1281125480000L
10  7248952  1281124588000L
11  7248952  1281124587000L
12  7248952  1281124561000L
13  7248952  1281124545000L
14  7248952  1281124522000L
15  7248952  1281124482000L
16  7248952  1281124456000L
17  7248952  1281124350000L
18  7248952  1281124330000L
19  7248952  1281124015000L


In [54]:
trad_features = build_temporal_features(trad_tweets)

print("Accounts:", len(trad_features))
print("Shape:", trad_features.shape)
print("\nFirst rows:")
print(trad_features.head())

C:\Users\namit\AppData\Local\Temp\ipykernel_1832\176108972.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates = pd.to_datetime(


Accounts: 1000
Shape: (1000, 12)

First rows:
    user_id  mean_intertweet_seconds  median_intertweet_seconds  \
0   7248952             79276.919714                     1609.5   
1   7732472            134433.280884                    22997.0   
2   9524952             72479.873521                    21670.0   
3  10788822              7010.893092                       83.0   
4  14596967             24271.258599                     3521.0   

   intertweet_cv  burstiness  posting_time_entropy  active_hour_count  \
0      10.545250    0.826769              3.574847                 20   
1       9.655149    0.812297              3.802020                 24   
2       2.604003    0.445062              4.442701                 24   
3       5.790071    0.705452              4.463715                 24   
4       6.961493    0.748791              4.524466                 24   

   night_activity_ratio  weekend_activity_ratio  top_hour_concentration  \
0              0.176330              

In [59]:
all_temporal_features = {}

for account_type, path in dataset_paths.items():
    print("\n" + "=" * 60)
    print("Processing:", account_type)

    tweets = pd.read_csv(
        path,
        encoding="latin1"
    )

    print("Tweets:", len(tweets))

    features = build_temporal_features(tweets)
    features["account_type"] = account_type

    all_temporal_features[account_type] = features

    print("Accounts:", len(features))
    print("Feature shape:", features.shape)

    del tweets


Processing: fake_followers


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\1138504534.py:7: DtypeWarning: Columns (0: in_reply_to_screen_name, 1: place) have mixed types. Specify dtype option on import or set low_memory=False.
  tweets = pd.read_csv(
C:\Users\namit\AppData\Local\Temp\ipykernel_1832\1676061094.py:56: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates.loc[normal_mask] = pd.to_datetime(


Tweets: 196027
Accounts: 3202
Feature shape: (3202, 13)

Processing: genuine_accounts


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\1138504534.py:7: DtypeWarning: Columns (0: id) have mixed types. Specify dtype option on import or set low_memory=False.
  tweets = pd.read_csv(


Tweets: 2839362


...

Accounts: 3457
Feature shape: (3457, 13)

Processing: social_spambots_3


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\1138504534.py:7: DtypeWarning: Columns (0: in_reply_to_screen_name, 1: place) have mixed types. Specify dtype option on import or set low_memory=False.
  tweets = pd.read_csv(


Tweets: 1418557


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\1676061094.py:56: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates.loc[normal_mask] = pd.to_datetime(


Accounts: 464
Feature shape: (464, 13)

Processing: traditional_spambots_1
Tweets: 145094
Accounts: 1000
Feature shape: (1000, 13)


In [60]:
temporal_dataset = pd.concat(
    all_temporal_features.values(),
    ignore_index=True
)

print("Final temporal dataset shape:", temporal_dataset.shape)

print("\nAccounts by category:")
print(temporal_dataset["account_type"].value_counts())

print("\nDuplicate user IDs:",
      temporal_dataset["user_id"].duplicated().sum())

Final temporal dataset shape: (10197, 13)

Accounts by category:
account_type
social_spambots_2         3457
fake_followers            3202
genuine_accounts          1083
traditional_spambots_1    1000
social_spambots_1          991
social_spambots_3          464
Name: count, dtype: int64

Duplicate user IDs: 0


In [61]:
label_map = {
    "fake_followers": 1,
    "genuine_accounts": 0,
    "social_spambots_1": 1,
    "social_spambots_2": 1,
    "social_spambots_3": 1,
    "traditional_spambots_1": 1
}

temporal_dataset["label"] = (
    temporal_dataset["account_type"].map(label_map)
)

print(temporal_dataset.shape)
print(temporal_dataset["label"].value_counts())

(10197, 14)
label
1    9114
0    1083
Name: count, dtype: int64


In [62]:
# Check temporal dataset structure
print("Shape:", temporal_dataset.shape)

print("\nColumns:")
print(temporal_dataset.columns.tolist())

print("\nMissing values:")
print(temporal_dataset.isna().sum())

print("\nDuplicate user IDs:")
print(temporal_dataset["user_id"].duplicated().sum())

Shape: (10197, 14)

Columns:
['user_id', 'mean_intertweet_seconds', 'median_intertweet_seconds', 'intertweet_cv', 'burstiness', 'posting_time_entropy', 'active_hour_count', 'night_activity_ratio', 'weekend_activity_ratio', 'top_hour_concentration', 'daily_activity_cv', 'tweet_count', 'account_type', 'label']

Missing values:
user_id                        0
mean_intertweet_seconds      363
median_intertweet_seconds    363
intertweet_cv                538
burstiness                   538
posting_time_entropy           0
active_hour_count              0
night_activity_ratio           0
weekend_activity_ratio         0
top_hour_concentration         0
daily_activity_cv            396
tweet_count                    0
account_type                   0
label                          0
dtype: int64

Duplicate user IDs:
0


In [63]:
# Check correlations and redundancy among temporal features

temporal_numeric = temporal_dataset.select_dtypes(include="number")

print("Correlation with label:")
print(
    temporal_numeric.corr()["label"]
    .drop("label")
    .sort_values(ascending=False)
)

print("\nCorrelation between temporal features:")
print(
    temporal_numeric.drop(columns=["label"])
    .corr()
    .round(3)
)

Correlation with label:
user_id                      0.172386
top_hour_concentration       0.146011
mean_intertweet_seconds      0.055061
weekend_activity_ratio       0.045297
median_intertweet_seconds    0.034172
intertweet_cv               -0.126055
daily_activity_cv           -0.152829
burstiness                  -0.155744
night_activity_ratio        -0.157114
posting_time_entropy        -0.213778
active_hour_count           -0.424636
tweet_count                 -0.618137
Name: label, dtype: float64

Correlation between temporal features:
                           user_id  mean_intertweet_seconds  \
user_id                      1.000                   -0.142   
mean_intertweet_seconds     -0.142                    1.000   
median_intertweet_seconds   -0.090                    0.921   
intertweet_cv                0.175                   -0.124   
burstiness                   0.385                   -0.149   
posting_time_entropy         0.174                   -0.285   
active_hour

In [64]:
# Compare temporal feature distributions by class

temporal_features_for_analysis = [
    "mean_intertweet_seconds",
    "median_intertweet_seconds",
    "intertweet_cv",
    "burstiness",
    "posting_time_entropy",
    "active_hour_count",
    "night_activity_ratio",
    "weekend_activity_ratio",
    "top_hour_concentration",
    "daily_activity_cv",
]

print(
    temporal_dataset.groupby("label")[temporal_features_for_analysis]
    .agg(["mean", "median"])
    .round(4)
)

      mean_intertweet_seconds              median_intertweet_seconds          \
                         mean       median                      mean  median   
label                                                                          
0                  72779.0337   12086.1993                11385.8149   694.0   
1                 543099.2890  119612.1648               277352.8302  3746.5   

      intertweet_cv         burstiness         posting_time_entropy          \
               mean  median       mean  median                 mean  median   
label                                                                         
0            3.9767  3.0323     0.5104  0.5040               4.1060  4.1489   
1            2.9864  3.3496     0.3744  0.5402               3.4501  3.6842   

      active_hour_count        night_activity_ratio          \
                   mean median                 mean  median   
label                                                         
0             

In [65]:
temporal_feature_cols = [
    "mean_intertweet_seconds",
    "intertweet_cv",
    "posting_time_entropy",
    "active_hour_count",
    "night_activity_ratio",
    "weekend_activity_ratio",
    "top_hour_concentration",
    "daily_activity_cv"
]

In [67]:
print("Behavior columns:")
print(behavior_df.columns.tolist())

print("\nTemporal columns:")
print(temporal_dataset.columns.tolist())

Behavior columns:
['id', 'account_type', 'label', 'statuses_count', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'created_at', 'updated', 'account_age_days', 'statuses_per_day', 'followers_friends_ratio', 'friends_followers_ratio', 'favorites_per_status', 'log_statuses_count', 'log_followers_count', 'log_friends_count', 'log_favourites_count', 'log_listed_count']

Temporal columns:
['user_id', 'mean_intertweet_seconds', 'median_intertweet_seconds', 'intertweet_cv', 'burstiness', 'posting_time_entropy', 'active_hour_count', 'night_activity_ratio', 'weekend_activity_ratio', 'top_hour_concentration', 'daily_activity_cv', 'tweet_count', 'account_type', 'label']


In [68]:
# ============================================================
# STEP 1 — Align IDs and merge temporal + behavioral features
# ============================================================

# Make a copy so the original behavioral dataset stays untouched
behavior_for_merge = behavior_df.rename(
    columns={"id": "user_id"}
).copy()

# Select only the temporal features we want to test
temporal_selected = temporal_dataset[
    ["user_id", "account_type", "label"] + temporal_feature_cols
].copy()

# Merge
combined_temporal_behavior = behavior_for_merge.merge(
    temporal_selected,
    on="user_id",
    how="inner",
    suffixes=("_behavior", "_temporal")
)

print("Combined shape:", combined_temporal_behavior.shape)

print("\nColumns:")
print(combined_temporal_behavior.columns.tolist())

print("\nAccount types:")
print(
    combined_temporal_behavior["account_type_temporal"]
    .value_counts()
)

print("\nDuplicate IDs:")
print(
    combined_temporal_behavior["user_id"].duplicated().sum()
)

Combined shape: (10197, 30)

Columns:
['user_id', 'account_type_behavior', 'label_behavior', 'statuses_count', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'created_at', 'updated', 'account_age_days', 'statuses_per_day', 'followers_friends_ratio', 'friends_followers_ratio', 'favorites_per_status', 'log_statuses_count', 'log_followers_count', 'log_friends_count', 'log_favourites_count', 'log_listed_count', 'account_type_temporal', 'label_temporal', 'mean_intertweet_seconds', 'intertweet_cv', 'posting_time_entropy', 'active_hour_count', 'night_activity_ratio', 'weekend_activity_ratio', 'top_hour_concentration', 'daily_activity_cv']

Account types:
account_type_temporal
social_spambots_2         3457
fake_followers            3202
genuine_accounts          1083
traditional_spambots_1    1000
social_spambots_1          991
social_spambots_3          464
Name: count, dtype: int64

Duplicate IDs:
0


In [69]:
# ============================================================
# STEP 2 — Leave-One-Bot-Type-Out split
# Unseen bot type = Fake Followers
# ============================================================

UNSEEN_TYPE = "fake_followers"

# Training pool: everything except Fake Followers
lobo_train = combined_temporal_behavior[
    combined_temporal_behavior["account_type_temporal"] != UNSEEN_TYPE
].copy()

# Test pool: Fake Followers + Genuine Accounts
lobo_test = combined_temporal_behavior[
    combined_temporal_behavior["account_type_temporal"].isin(
        ["fake_followers", "genuine_accounts"]
    )
].copy()

print("LOBO train shape:", lobo_train.shape)
print("LOBO test shape:", lobo_test.shape)

print("\nTraining account types:")
print(lobo_train["account_type_temporal"].value_counts())

print("\nTest account types:")
print(lobo_test["account_type_temporal"].value_counts())

print("\nTraining labels:")
print(lobo_train["label_temporal"].value_counts())

print("\nTest labels:")
print(lobo_test["label_temporal"].value_counts())

LOBO train shape: (6995, 30)
LOBO test shape: (4285, 30)

Training account types:
account_type_temporal
social_spambots_2         3457
genuine_accounts          1083
traditional_spambots_1    1000
social_spambots_1          991
social_spambots_3          464
Name: count, dtype: int64

Test account types:
account_type_temporal
fake_followers      3202
genuine_accounts    1083
Name: count, dtype: int64

Training labels:
label_temporal
1    5912
0    1083
Name: count, dtype: int64

Test labels:
label_temporal
1    3202
0    1083
Name: count, dtype: int64


In [70]:
# ============================================================
# STEP 3 — Prepare 15 Behavioral + 8 Temporal Features
# ============================================================

behavior_feature_cols = [
    "statuses_count",
    "followers_count",
    "friends_count",
    "favourites_count",
    "listed_count",
    "account_age_days",
    "statuses_per_day",
    "followers_friends_ratio",
    "friends_followers_ratio",
    "favorites_per_status",
    "log_statuses_count",
    "log_followers_count",
    "log_friends_count",
    "log_favourites_count",
    "log_listed_count"
]

temporal_feature_cols = [
    "mean_intertweet_seconds",
    "intertweet_cv",
    "posting_time_entropy",
    "active_hour_count",
    "night_activity_ratio",
    "weekend_activity_ratio",
    "top_hour_concentration",
    "daily_activity_cv"
]

all_feature_cols = behavior_feature_cols + temporal_feature_cols

print("Behavioral features:", len(behavior_feature_cols))
print("Temporal features:", len(temporal_feature_cols))
print("Total features:", len(all_feature_cols))

Behavioral features: 15
Temporal features: 8
Total features: 23


In [71]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Features
X_lobo = lobo_train[all_feature_cols].copy()
y_lobo = lobo_train["label_temporal"].values

X_test_temporal = lobo_test[all_feature_cols].copy()
y_test_temporal = lobo_test["label_temporal"].values

# ------------------------------------------------------------
# Train / validation split
# ------------------------------------------------------------

X_train_temp, X_val_temp, y_train_temp, y_val_temp = train_test_split(
    X_lobo,
    y_lobo,
    test_size=0.20,
    random_state=42,
    stratify=y_lobo
)

print("Train:", X_train_temp.shape)
print("Validation:", X_val_temp.shape)
print("Test:", X_test_temporal.shape)

# ------------------------------------------------------------
# Imputation — FIT ONLY ON TRAINING DATA
# ------------------------------------------------------------

temporal_imputer = SimpleImputer(strategy="median")

X_train_temp_imp = temporal_imputer.fit_transform(X_train_temp)
X_val_temp_imp = temporal_imputer.transform(X_val_temp)
X_test_temp_imp = temporal_imputer.transform(X_test_temporal)

# ------------------------------------------------------------
# Scaling — FIT ONLY ON TRAINING DATA
# ------------------------------------------------------------

temporal_scaler = StandardScaler()

X_train_temp_scaled = temporal_scaler.fit_transform(X_train_temp_imp)
X_val_temp_scaled = temporal_scaler.transform(X_val_temp_imp)
X_test_temp_scaled = temporal_scaler.transform(X_test_temp_imp)

print("\nMissing values after imputation:")
print("Train:", np.isnan(X_train_temp_scaled).sum())
print("Validation:", np.isnan(X_val_temp_scaled).sum())
print("Test:", np.isnan(X_test_temp_scaled).sum())

Train: (5596, 23)
Validation: (1399, 23)
Test: (4285, 23)

Missing values after imputation:
Train: 0
Validation: 0
Test: 0


In [72]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ============================================================
# STEP 4 — Temporal-Enhanced Neural Network
# ============================================================

tf.random.set_seed(42)
np.random.seed(42)

# Class weights — calculated ONLY from training labels
classes = np.unique(y_train_temp)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_temp
)

temporal_class_weights = dict(
    zip(classes, class_weights_values)
)

print("Class weights:", temporal_class_weights)

# ------------------------------------------------------------
# Model
# Same architecture as behavioral baseline
# ------------------------------------------------------------

temporal_nn = Sequential([
    Input(shape=(X_train_temp_scaled.shape[1],)),

    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.25),

    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.20),

    Dense(32, activation="relu"),
    Dropout(0.10),

    Dense(1, activation="sigmoid")
])

temporal_nn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# ------------------------------------------------------------
# Callbacks
# ------------------------------------------------------------

early_stopping_temporal = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr_temporal = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

history_temporal = temporal_nn.fit(
    X_train_temp_scaled,
    y_train_temp,
    validation_data=(X_val_temp_scaled, y_val_temp),
    epochs=50,
    batch_size=32,
    class_weight=temporal_class_weights,
    callbacks=[
        early_stopping_temporal,
        reduce_lr_temporal
    ],
    verbose=1
)

Class weights: {np.int64(0): np.float64(3.23094688221709), np.int64(1): np.float64(0.5915433403805497)}
Epoch 1/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9287 - loss: 0.1800 - val_accuracy: 0.9929 - val_loss: 0.0721 - learning_rate: 0.0010
Epoch 2/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9818 - loss: 0.0739 - val_accuracy: 0.9907 - val_loss: 0.0483 - learning_rate: 0.0010
Epoch 3/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9839 - loss: 0.0622 - val_accuracy: 0.9893 - val_loss: 0.0445 - learning_rate: 0.0010
Epoch 4/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9820 - loss: 0.0528 - val_accuracy: 0.9900 - val_loss: 0.0432 - learning_rate: 0.0010
Epoch 5/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9853 - loss: 0.0431 - val_accuracy: 0.9878 - val_loss: 0.0433 - learning_rate: 0.0010
Epoch 6/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9877 - loss: 0.0411 - val_accuracy: 0.9893 - val_loss: 0.0453 - learni

In [73]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ============================================================
# STEP 5 — Evaluate on unseen Fake Followers
# ============================================================

temporal_test_prob = temporal_nn.predict(
    X_test_temp_scaled,
    verbose=0
).ravel()

temporal_test_pred = (
    temporal_test_prob >= 0.5
).astype(int)

temporal_acc = accuracy_score(
    y_test_temporal,
    temporal_test_pred
)

temporal_precision = precision_score(
    y_test_temporal,
    temporal_test_pred
)

temporal_recall = recall_score(
    y_test_temporal,
    temporal_test_pred
)

temporal_f1 = f1_score(
    y_test_temporal,
    temporal_test_pred
)

temporal_cm = confusion_matrix(
    y_test_temporal,
    temporal_test_pred
)

print("Temporal-Enhanced LOBO Results")
print("=" * 45)
print(f"Accuracy :  {temporal_acc * 100:.2f}%")
print(f"Precision:  {temporal_precision * 100:.2f}%")
print(f"Recall   :  {temporal_recall * 100:.2f}%")
print(f"F1 Score :  {temporal_f1 * 100:.2f}%")

print("\nConfusion Matrix:")
print(temporal_cm)

Temporal-Enhanced LOBO Results
Accuracy :  98.18%
Precision:  99.81%
Recall   :  97.75%
F1 Score :  98.77%

Confusion Matrix:
[[1077    6]
 [  72 3130]]


In [74]:
# ============================================================
# TEMPORAL ABLATION 1: INTERTWEET TIMING
# ============================================================

temporal_ablation_cols = [
    "mean_intertweet_seconds",
    "intertweet_cv"
]

X_ablation = lobo_train[temporal_ablation_cols].copy()
X_ablation_test = lobo_test[temporal_ablation_cols].copy()

# Same train/validation split as before
X_train_ab, X_val_ab, y_train_ab, y_val_ab = train_test_split(
    X_ablation,
    y_lobo,
    test_size=0.20,
    random_state=42,
    stratify=y_lobo
)

# Training-only imputation
ab_imputer = SimpleImputer(strategy="median")

X_train_ab_imp = ab_imputer.fit_transform(X_train_ab)
X_val_ab_imp = ab_imputer.transform(X_val_ab)
X_test_ab_imp = ab_imputer.transform(X_ablation_test)

# Training-only scaling
ab_scaler = StandardScaler()

X_train_ab_scaled = ab_scaler.fit_transform(X_train_ab_imp)
X_val_ab_scaled = ab_scaler.transform(X_val_ab_imp)
X_test_ab_scaled = ab_scaler.transform(X_test_ab_imp)

# Class weights
classes = np.unique(y_train_ab)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_ab
)

ab_class_weights = dict(zip(classes, class_weights_values))

print("Features:", temporal_ablation_cols)
print("Class weights:", ab_class_weights)

# Same architecture for fair comparison
tf.random.set_seed(42)
np.random.seed(42)

ablation_nn = Sequential([
    Input(shape=(X_train_ab_scaled.shape[1],)),
    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.25),
    
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.20),
    
    Dense(32, activation="relu"),
    Dropout(0.10),
    
    Dense(1, activation="sigmoid")
])

ablation_nn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping_ab = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr_ab = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

history_ablation = ablation_nn.fit(
    X_train_ab_scaled,
    y_train_ab,
    validation_data=(X_val_ab_scaled, y_val_ab),
    epochs=50,
    batch_size=32,
    class_weight=ab_class_weights,
    callbacks=[
        early_stopping_ab,
        reduce_lr_ab
    ],
    verbose=1
)

Features: ['mean_intertweet_seconds', 'intertweet_cv']
Class weights: {np.int64(0): np.float64(3.23094688221709), np.int64(1): np.float64(0.5915433403805497)}
Epoch 1/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.7227 - loss: 0.5043 - val_accuracy: 0.1680 - val_loss: 0.8312 - learning_rate: 0.0010
Epoch 2/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7622 - loss: 0.4258 - val_accuracy: 0.6376 - val_loss: 0.7298 - learning_rate: 0.0010
Epoch 3/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.7736 - loss: 0.4097 - val_accuracy: 0.7255 - val_loss: 0.5143 - learning_rate: 0.0010
Epoch 4/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7902 - loss: 0.3890 - val_accuracy: 0.8420 - val_loss: 0.3575 - learning_rate: 0.0010
Epoch 5/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8056 - loss: 0.3827 - val_accuracy: 0.8442 - val_loss: 0.3618 - learning_rate: 0.0010
Epoch 6/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8077 - loss: 0

In [75]:
# ============================================================
# EVALUATION
# ============================================================

ab_test_prob = ablation_nn.predict(
    X_test_ab_scaled,
    verbose=0
).ravel()

ab_test_pred = (ab_test_prob >= 0.5).astype(int)

ab_acc = accuracy_score(y_test_temporal, ab_test_pred)
ab_precision = precision_score(y_test_temporal, ab_test_pred)
ab_recall = recall_score(y_test_temporal, ab_test_pred)
ab_f1 = f1_score(y_test_temporal, ab_test_pred)
ab_cm = confusion_matrix(y_test_temporal, ab_test_pred)

print("\nTEMPORAL ABLATION 1 — INTERTWEET TIMING")
print("=" * 55)
print(f"Accuracy :  {ab_acc * 100:.2f}%")
print(f"Precision:  {ab_precision * 100:.2f}%")
print(f"Recall   :  {ab_recall * 100:.2f}%")
print(f"F1 Score :  {ab_f1 * 100:.2f}%")

print("\nConfusion Matrix:")
print(ab_cm)


TEMPORAL ABLATION 1 — INTERTWEET TIMING
Accuracy :  90.25%
Precision:  96.22%
Recall   :  90.51%
F1 Score :  93.27%

Confusion Matrix:
[[ 969  114]
 [ 304 2898]]


In [76]:
temporal_ablation_cols = [
    "posting_time_entropy",
    "active_hour_count",
    "top_hour_concentration"
]

In [77]:
# ============================================================
# TEMPORAL ABLATION 2: POSTING-TIME BEHAVIOR
# ============================================================

temporal_ablation_cols = [
    "posting_time_entropy",
    "active_hour_count",
    "top_hour_concentration"
]

X_ablation = lobo_train[temporal_ablation_cols].copy()
X_ablation_test = lobo_test[temporal_ablation_cols].copy()

X_train_ab, X_val_ab, y_train_ab, y_val_ab = train_test_split(
    X_ablation,
    y_lobo,
    test_size=0.20,
    random_state=42,
    stratify=y_lobo
)

# Training-only imputation
ab_imputer = SimpleImputer(strategy="median")

X_train_ab_imp = ab_imputer.fit_transform(X_train_ab)
X_val_ab_imp = ab_imputer.transform(X_val_ab)
X_test_ab_imp = ab_imputer.transform(X_ablation_test)

# Training-only scaling
ab_scaler = StandardScaler()

X_train_ab_scaled = ab_scaler.fit_transform(X_train_ab_imp)
X_val_ab_scaled = ab_scaler.transform(X_val_ab_imp)
X_test_ab_scaled = ab_scaler.transform(X_test_ab_imp)

# Class weights
classes = np.unique(y_train_ab)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_ab
)

ab_class_weights = dict(zip(classes, class_weights_values))

print("Features:", temporal_ablation_cols)
print("Class weights:", ab_class_weights)

# Same architecture for fair comparison
tf.random.set_seed(42)
np.random.seed(42)

ablation_nn = Sequential([
    Input(shape=(X_train_ab_scaled.shape[1],)),
    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.25),

    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.20),

    Dense(32, activation="relu"),
    Dropout(0.10),

    Dense(1, activation="sigmoid")
])

ablation_nn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping_ab = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr_ab = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

history_ablation = ablation_nn.fit(
    X_train_ab_scaled,
    y_train_ab,
    validation_data=(X_val_ab_scaled, y_val_ab),
    epochs=50,
    batch_size=32,
    class_weight=ab_class_weights,
    callbacks=[
        early_stopping_ab,
        reduce_lr_ab
    ],
    verbose=1
)

Features: ['posting_time_entropy', 'active_hour_count', 'top_hour_concentration']
Class weights: {np.int64(0): np.float64(3.23094688221709), np.int64(1): np.float64(0.5915433403805497)}
Epoch 1/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.8972 - loss: 0.2871 - val_accuracy: 0.8313 - val_loss: 0.5566 - learning_rate: 0.0010
Epoch 2/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9199 - loss: 0.2476 - val_accuracy: 0.9257 - val_loss: 0.3502 - learning_rate: 0.0010
Epoch 3/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9228 - loss: 0.2362 - val_accuracy: 0.9364 - val_loss: 0.2259 - learning_rate: 0.0010
Epoch 4/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.9241 - loss: 0.2342 - val_accuracy: 0.9371 - val_loss: 0.1866 - learning_rate: 0.0010
Epoch 5/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9237 - loss: 0.2353 - val_accuracy: 0.9392 - val_loss: 0.1819 - learning_rate: 0.0010
Epoch 6/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step -

In [78]:
# ============================================================
# EVALUATION
# ============================================================

ab_test_prob = ablation_nn.predict(
    X_test_ab_scaled,
    verbose=0
).ravel()

ab_test_pred = (ab_test_prob >= 0.5).astype(int)

ab_acc = accuracy_score(y_test_temporal, ab_test_pred)
ab_precision = precision_score(y_test_temporal, ab_test_pred)
ab_recall = recall_score(y_test_temporal, ab_test_pred)
ab_f1 = f1_score(y_test_temporal, ab_test_pred)
ab_cm = confusion_matrix(y_test_temporal, ab_test_pred)

print("\nTEMPORAL ABLATION 2 — POSTING-TIME BEHAVIOR")
print("=" * 55)
print(f"Accuracy :  {ab_acc * 100:.2f}%")
print(f"Precision:  {ab_precision * 100:.2f}%")
print(f"Recall   :  {ab_recall * 100:.2f}%")
print(f"F1 Score :  {ab_f1 * 100:.2f}%")

print("\nConfusion Matrix:")
print(ab_cm)


TEMPORAL ABLATION 2 — POSTING-TIME BEHAVIOR
Accuracy :  88.68%
Precision:  97.62%
Recall   :  86.98%
F1 Score :  91.99%

Confusion Matrix:
[[1015   68]
 [ 417 2785]]


In [79]:
# ============================================================
# TEMPORAL ABLATION 3: ACTIVITY VARIABILITY
# ============================================================

temporal_ablation_cols = [
    "daily_activity_cv",
    "night_activity_ratio",
    "weekend_activity_ratio"
]

X_ablation = lobo_train[temporal_ablation_cols].copy()
X_ablation_test = lobo_test[temporal_ablation_cols].copy()

X_train_ab, X_val_ab, y_train_ab, y_val_ab = train_test_split(
    X_ablation,
    y_lobo,
    test_size=0.20,
    random_state=42,
    stratify=y_lobo
)

# Training-only imputation
ab_imputer = SimpleImputer(strategy="median")

X_train_ab_imp = ab_imputer.fit_transform(X_train_ab)
X_val_ab_imp = ab_imputer.transform(X_val_ab)
X_test_ab_imp = ab_imputer.transform(X_ablation_test)

# Training-only scaling
ab_scaler = StandardScaler()

X_train_ab_scaled = ab_scaler.fit_transform(X_train_ab_imp)
X_val_ab_scaled = ab_scaler.transform(X_val_ab_imp)
X_test_ab_scaled = ab_scaler.transform(X_test_ab_imp)

# Class weights
classes = np.unique(y_train_ab)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_ab
)

ab_class_weights = dict(zip(classes, class_weights_values))

print("Features:", temporal_ablation_cols)
print("Class weights:", ab_class_weights)

# Same architecture
tf.random.set_seed(42)
np.random.seed(42)

ablation_nn = Sequential([
    Input(shape=(X_train_ab_scaled.shape[1],)),
    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.25),

    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.20),

    Dense(32, activation="relu"),
    Dropout(0.10),

    Dense(1, activation="sigmoid")
])

ablation_nn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping_ab = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr_ab = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

history_ablation = ablation_nn.fit(
    X_train_ab_scaled,
    y_train_ab,
    validation_data=(X_val_ab_scaled, y_val_ab),
    epochs=50,
    batch_size=32,
    class_weight=ab_class_weights,
    callbacks=[
        early_stopping_ab,
        reduce_lr_ab
    ],
    verbose=1
)

Features: ['daily_activity_cv', 'night_activity_ratio', 'weekend_activity_ratio']
Class weights: {np.int64(0): np.float64(3.23094688221709), np.int64(1): np.float64(0.5915433403805497)}
Epoch 1/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7441 - loss: 0.6072 - val_accuracy: 0.7613 - val_loss: 0.5808 - learning_rate: 0.0010
Epoch 2/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7387 - loss: 0.5359 - val_accuracy: 0.7691 - val_loss: 0.5175 - learning_rate: 0.0010
Epoch 3/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7470 - loss: 0.5165 - val_accuracy: 0.7791 - val_loss: 0.4639 - learning_rate: 0.0010
Epoch 4/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.7575 - loss: 0.5009 - val_accuracy: 0.7884 - val_loss: 0.4270 - learning_rate: 0.0010
Epoch 5/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.7625 - loss: 0.4930 - val_accuracy: 0.7977 - val_loss: 0.4174 - learning_rate: 0.0010
Epoch 6/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step 

In [80]:
# ============================================================
# EVALUATION
# ============================================================

ab_test_prob = ablation_nn.predict(
    X_test_ab_scaled,
    verbose=0
).ravel()

ab_test_pred = (ab_test_prob >= 0.5).astype(int)

ab_acc = accuracy_score(y_test_temporal, ab_test_pred)
ab_precision = precision_score(y_test_temporal, ab_test_pred)
ab_recall = recall_score(y_test_temporal, ab_test_pred)
ab_f1 = f1_score(y_test_temporal, ab_test_pred)
ab_cm = confusion_matrix(y_test_temporal, ab_test_pred)

print("\nTEMPORAL ABLATION 3 — ACTIVITY VARIABILITY")
print("=" * 55)
print(f"Accuracy :  {ab_acc * 100:.2f}%")
print(f"Precision:  {ab_precision * 100:.2f}%")
print(f"Recall   :  {ab_recall * 100:.2f}%")
print(f"F1 Score :  {ab_f1 * 100:.2f}%")

print("\nConfusion Matrix:")
print(ab_cm)


TEMPORAL ABLATION 3 — ACTIVITY VARIABILITY
Accuracy :  82.38%
Precision:  92.47%
Recall   :  83.20%
F1 Score :  87.59%

Confusion Matrix:
[[ 866  217]
 [ 538 2664]]


In [81]:
# ============================================================
# TEMPORAL ABLATION 4
# BEHAVIORAL + POSTING-TIME FEATURES
# ============================================================

ablation_feature_cols = behavior_feature_cols + [
    "posting_time_entropy",
    "active_hour_count",
    "top_hour_concentration"
]

X_ablation = lobo_train[ablation_feature_cols].copy()
X_ablation_test = lobo_test[ablation_feature_cols].copy()

print("Total features:", len(ablation_feature_cols))
print("Features:", ablation_feature_cols)

# Same LOBO train/validation split
X_train_ab, X_val_ab, y_train_ab, y_val_ab = train_test_split(
    X_ablation,
    y_lobo,
    test_size=0.20,
    random_state=42,
    stratify=y_lobo
)

# Training-only imputation
ab_imputer = SimpleImputer(strategy="median")

X_train_ab_imp = ab_imputer.fit_transform(X_train_ab)
X_val_ab_imp = ab_imputer.transform(X_val_ab)
X_test_ab_imp = ab_imputer.transform(X_ablation_test)

# Training-only scaling
ab_scaler = StandardScaler()

X_train_ab_scaled = ab_scaler.fit_transform(X_train_ab_imp)
X_val_ab_scaled = ab_scaler.transform(X_val_ab_imp)
X_test_ab_scaled = ab_scaler.transform(X_test_ab_imp)

# Class weights
classes = np.unique(y_train_ab)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_ab
)

ab_class_weights = dict(zip(classes, class_weights_values))

print("Class weights:", ab_class_weights)

# Same NN architecture
tf.random.set_seed(42)
np.random.seed(42)

ablation_nn = Sequential([
    Input(shape=(X_train_ab_scaled.shape[1],)),

    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.25),

    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.20),

    Dense(32, activation="relu"),
    Dropout(0.10),

    Dense(1, activation="sigmoid")
])

ablation_nn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping_ab = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr_ab = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

history_ablation = ablation_nn.fit(
    X_train_ab_scaled,
    y_train_ab,
    validation_data=(X_val_ab_scaled, y_val_ab),
    epochs=50,
    batch_size=32,
    class_weight=ab_class_weights,
    callbacks=[
        early_stopping_ab,
        reduce_lr_ab
    ],
    verbose=1
)

Total features: 18
Features: ['statuses_count', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'account_age_days', 'statuses_per_day', 'followers_friends_ratio', 'friends_followers_ratio', 'favorites_per_status', 'log_statuses_count', 'log_followers_count', 'log_friends_count', 'log_favourites_count', 'log_listed_count', 'posting_time_entropy', 'active_hour_count', 'top_hour_concentration']
Class weights: {np.int64(0): np.float64(3.23094688221709), np.int64(1): np.float64(0.5915433403805497)}
Epoch 1/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9535 - loss: 0.1547 - val_accuracy: 0.9886 - val_loss: 0.0716 - learning_rate: 0.0010
Epoch 2/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9807 - loss: 0.0780 - val_accuracy: 0.9900 - val_loss: 0.0467 - learning_rate: 0.0010
Epoch 3/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9843 - loss: 0.0609 - val_accuracy: 0.9878 - val_loss: 0.0460 - learning_rate: 0.0010
Epoch 4/50
175/175 ━━

In [82]:
# ============================================================
# EVALUATION
# ============================================================

ab_test_prob = ablation_nn.predict(
    X_test_ab_scaled,
    verbose=0
).ravel()

ab_test_pred = (ab_test_prob >= 0.5).astype(int)

ab_acc = accuracy_score(y_test_temporal, ab_test_pred)
ab_precision = precision_score(y_test_temporal, ab_test_pred)
ab_recall = recall_score(y_test_temporal, ab_test_pred)
ab_f1 = f1_score(y_test_temporal, ab_test_pred)
ab_cm = confusion_matrix(y_test_temporal, ab_test_pred)

print("\nTEMPORAL ABLATION 4 — BEHAVIOR + POSTING-TIME")
print("=" * 60)
print(f"Accuracy :  {ab_acc * 100:.2f}%")
print(f"Precision:  {ab_precision * 100:.2f}%")
print(f"Recall   :  {ab_recall * 100:.2f}%")
print(f"F1 Score :  {ab_f1 * 100:.2f}%")

print("\nConfusion Matrix:")
print(ab_cm)


TEMPORAL ABLATION 4 — BEHAVIOR + POSTING-TIME
Accuracy :  98.69%
Precision:  99.68%
Recall   :  98.56%
F1 Score :  99.12%

Confusion Matrix:
[[1073   10]
 [  46 3156]]


In [83]:
# ============================================================
# TEMPORAL ABLATION 5
# BEHAVIOR + INTERTWEET TIMING
# ============================================================

ablation_feature_cols = behavior_feature_cols + [
    "mean_intertweet_seconds",
    "intertweet_cv"
]

X_ablation = lobo_train[ablation_feature_cols].copy()
X_ablation_test = lobo_test[ablation_feature_cols].copy()

print("Total features:", len(ablation_feature_cols))
print("Features:", ablation_feature_cols)

# Same LOBO train/validation split
X_train_ab, X_val_ab, y_train_ab, y_val_ab = train_test_split(
    X_ablation,
    y_lobo,
    test_size=0.20,
    random_state=42,
    stratify=y_lobo
)

# Training-only imputation
ab_imputer = SimpleImputer(strategy="median")

X_train_ab_imp = ab_imputer.fit_transform(X_train_ab)
X_val_ab_imp = ab_imputer.transform(X_val_ab)
X_test_ab_imp = ab_imputer.transform(X_ablation_test)

# Training-only scaling
ab_scaler = StandardScaler()

X_train_ab_scaled = ab_scaler.fit_transform(X_train_ab_imp)
X_val_ab_scaled = ab_scaler.transform(X_val_ab_imp)
X_test_ab_scaled = ab_scaler.transform(X_test_ab_imp)

# Class weights
classes = np.unique(y_train_ab)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_ab
)

ab_class_weights = dict(zip(classes, class_weights_values))

print("Class weights:", ab_class_weights)

# Same architecture
tf.random.set_seed(42)
np.random.seed(42)

ablation_nn = Sequential([
    Input(shape=(X_train_ab_scaled.shape[1],)),

    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.25),

    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.20),

    Dense(32, activation="relu"),
    Dropout(0.10),

    Dense(1, activation="sigmoid")
])

ablation_nn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping_ab = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr_ab = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

history_ablation = ablation_nn.fit(
    X_train_ab_scaled,
    y_train_ab,
    validation_data=(X_val_ab_scaled, y_val_ab),
    epochs=50,
    batch_size=32,
    class_weight=ab_class_weights,
    callbacks=[
        early_stopping_ab,
        reduce_lr_ab
    ],
    verbose=1
)

Total features: 17
Features: ['statuses_count', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'account_age_days', 'statuses_per_day', 'followers_friends_ratio', 'friends_followers_ratio', 'favorites_per_status', 'log_statuses_count', 'log_followers_count', 'log_friends_count', 'log_favourites_count', 'log_listed_count', 'mean_intertweet_seconds', 'intertweet_cv']
Class weights: {np.int64(0): np.float64(3.23094688221709), np.int64(1): np.float64(0.5915433403805497)}
Epoch 1/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9087 - loss: 0.1831 - val_accuracy: 0.9900 - val_loss: 0.0757 - learning_rate: 0.0010
Epoch 2/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9809 - loss: 0.0804 - val_accuracy: 0.9893 - val_loss: 0.0446 - learning_rate: 0.0010
Epoch 3/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9820 - loss: 0.0604 - val_accuracy: 0.9900 - val_loss: 0.0395 - learning_rate: 0.0010
Epoch 4/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms

In [84]:
# ============================================================
# EVALUATION
# ============================================================

ab_test_prob = ablation_nn.predict(
    X_test_ab_scaled,
    verbose=0
).ravel()

ab_test_pred = (ab_test_prob >= 0.5).astype(int)

ab_acc = accuracy_score(y_test_temporal, ab_test_pred)
ab_precision = precision_score(y_test_temporal, ab_test_pred)
ab_recall = recall_score(y_test_temporal, ab_test_pred)
ab_f1 = f1_score(y_test_temporal, ab_test_pred)
ab_cm = confusion_matrix(y_test_temporal, ab_test_pred)

print("\nTEMPORAL ABLATION 5 — BEHAVIOR + INTERTWEET TIMING")
print("=" * 60)
print(f"Accuracy :  {ab_acc * 100:.2f}%")
print(f"Precision:  {ab_precision * 100:.2f}%")
print(f"Recall   :  {ab_recall * 100:.2f}%")
print(f"F1 Score :  {ab_f1 * 100:.2f}%")

print("\nConfusion Matrix:")
print(ab_cm)


TEMPORAL ABLATION 5 — BEHAVIOR + INTERTWEET TIMING
Accuracy :  98.83%
Precision:  99.53%
Recall   :  98.91%
F1 Score :  99.22%

Confusion Matrix:
[[1068   15]
 [  35 3167]]


In [85]:
# ============================================================
# TEMPORAL ABLATION 6
# BEHAVIOR + ACTIVITY VARIABILITY
# ============================================================

ablation_feature_cols = behavior_feature_cols + [
    "daily_activity_cv",
    "night_activity_ratio",
    "weekend_activity_ratio"
]

X_ablation = lobo_train[ablation_feature_cols].copy()
X_ablation_test = lobo_test[ablation_feature_cols].copy()

print("Total features:", len(ablation_feature_cols))
print("Features:", ablation_feature_cols)

X_train_ab, X_val_ab, y_train_ab, y_val_ab = train_test_split(
    X_ablation,
    y_lobo,
    test_size=0.20,
    random_state=42,
    stratify=y_lobo
)

ab_imputer = SimpleImputer(strategy="median")

X_train_ab_imp = ab_imputer.fit_transform(X_train_ab)
X_val_ab_imp = ab_imputer.transform(X_val_ab)
X_test_ab_imp = ab_imputer.transform(X_ablation_test)

ab_scaler = StandardScaler()

X_train_ab_scaled = ab_scaler.fit_transform(X_train_ab_imp)
X_val_ab_scaled = ab_scaler.transform(X_val_ab_imp)
X_test_ab_scaled = ab_scaler.transform(X_test_ab_imp)

classes = np.unique(y_train_ab)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_ab
)

ab_class_weights = dict(zip(classes, class_weights_values))

print("Class weights:", ab_class_weights)

tf.random.set_seed(42)
np.random.seed(42)

ablation_nn = Sequential([
    Input(shape=(X_train_ab_scaled.shape[1],)),

    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.25),

    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.20),

    Dense(32, activation="relu"),
    Dropout(0.10),

    Dense(1, activation="sigmoid")
])

ablation_nn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping_ab = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr_ab = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

history_ablation = ablation_nn.fit(
    X_train_ab_scaled,
    y_train_ab,
    validation_data=(X_val_ab_scaled, y_val_ab),
    epochs=50,
    batch_size=32,
    class_weight=ab_class_weights,
    callbacks=[
        early_stopping_ab,
        reduce_lr_ab
    ],
    verbose=1
)

Total features: 18
Features: ['statuses_count', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'account_age_days', 'statuses_per_day', 'followers_friends_ratio', 'friends_followers_ratio', 'favorites_per_status', 'log_statuses_count', 'log_followers_count', 'log_friends_count', 'log_favourites_count', 'log_listed_count', 'daily_activity_cv', 'night_activity_ratio', 'weekend_activity_ratio']
Class weights: {np.int64(0): np.float64(3.23094688221709), np.int64(1): np.float64(0.5915433403805497)}
Epoch 1/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 12s 40ms/step - accuracy: 0.9248 - loss: 0.1695 - val_accuracy: 0.9914 - val_loss: 0.0851 - learning_rate: 0.0010
Epoch 2/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9805 - loss: 0.0675 - val_accuracy: 0.9900 - val_loss: 0.0435 - learning_rate: 0.0010
Epoch 3/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9832 - loss: 0.0522 - val_accuracy: 0.9893 - val_loss: 0.0459 - learning_rate: 0.0010
Epoch 4/50
175/175

In [86]:
# ============================================================
# EVALUATION
# ============================================================

ab_test_prob = ablation_nn.predict(
    X_test_ab_scaled,
    verbose=0
).ravel()

ab_test_pred = (ab_test_prob >= 0.5).astype(int)

ab_acc = accuracy_score(y_test_temporal, ab_test_pred)
ab_precision = precision_score(y_test_temporal, ab_test_pred)
ab_recall = recall_score(y_test_temporal, ab_test_pred)
ab_f1 = f1_score(y_test_temporal, ab_test_pred)
ab_cm = confusion_matrix(y_test_temporal, ab_test_pred)

print("\nTEMPORAL ABLATION 6 — BEHAVIOR + ACTIVITY VARIABILITY")
print("=" * 65)
print(f"Accuracy :  {ab_acc * 100:.2f}%")
print(f"Precision:  {ab_precision * 100:.2f}%")
print(f"Recall   :  {ab_recall * 100:.2f}%")
print(f"F1 Score :  {ab_f1 * 100:.2f}%")

print("\nConfusion Matrix:")
print(ab_cm)


TEMPORAL ABLATION 6 — BEHAVIOR + ACTIVITY VARIABILITY
Accuracy :  98.76%
Precision:  99.72%
Recall   :  98.63%
F1 Score :  99.17%

Confusion Matrix:
[[1074    9]
 [  44 3158]]


In [87]:
# ============================================================
# TEMPORAL ABLATION 7
# BEHAVIOR + INTERTWEET + ACTIVITY VARIABILITY
# ============================================================

ablation_feature_cols = behavior_feature_cols + [
    "mean_intertweet_seconds",
    "intertweet_cv",
    "daily_activity_cv",
    "night_activity_ratio",
    "weekend_activity_ratio"
]

X_ablation = lobo_train[ablation_feature_cols].copy()
X_ablation_test = lobo_test[ablation_feature_cols].copy()

print("Total features:", len(ablation_feature_cols))
print("Features:", ablation_feature_cols)

X_train_ab, X_val_ab, y_train_ab, y_val_ab = train_test_split(
    X_ablation,
    y_lobo,
    test_size=0.20,
    random_state=42,
    stratify=y_lobo
)

# Training-only imputation
ab_imputer = SimpleImputer(strategy="median")

X_train_ab_imp = ab_imputer.fit_transform(X_train_ab)
X_val_ab_imp = ab_imputer.transform(X_val_ab)
X_test_ab_imp = ab_imputer.transform(X_ablation_test)

# Training-only scaling
ab_scaler = StandardScaler()

X_train_ab_scaled = ab_scaler.fit_transform(X_train_ab_imp)
X_val_ab_scaled = ab_scaler.transform(X_val_ab_imp)
X_test_ab_scaled = ab_scaler.transform(X_test_ab_imp)

# Class weights
classes = np.unique(y_train_ab)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_ab
)

ab_class_weights = dict(zip(classes, class_weights_values))

print("Class weights:", ab_class_weights)

# Same architecture
tf.random.set_seed(42)
np.random.seed(42)

ablation_nn = Sequential([
    Input(shape=(X_train_ab_scaled.shape[1],)),

    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.25),

    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.20),

    Dense(32, activation="relu"),
    Dropout(0.10),

    Dense(1, activation="sigmoid")
])

ablation_nn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping_ab = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr_ab = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

history_ablation = ablation_nn.fit(
    X_train_ab_scaled,
    y_train_ab,
    validation_data=(X_val_ab_scaled, y_val_ab),
    epochs=50,
    batch_size=32,
    class_weight=ab_class_weights,
    callbacks=[
        early_stopping_ab,
        reduce_lr_ab
    ],
    verbose=1
)

Total features: 20
Features: ['statuses_count', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'account_age_days', 'statuses_per_day', 'followers_friends_ratio', 'friends_followers_ratio', 'favorites_per_status', 'log_statuses_count', 'log_followers_count', 'log_friends_count', 'log_favourites_count', 'log_listed_count', 'mean_intertweet_seconds', 'intertweet_cv', 'daily_activity_cv', 'night_activity_ratio', 'weekend_activity_ratio']
Class weights: {np.int64(0): np.float64(3.23094688221709), np.int64(1): np.float64(0.5915433403805497)}
Epoch 1/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9380 - loss: 0.1645 - val_accuracy: 0.9893 - val_loss: 0.0733 - learning_rate: 0.0010
Epoch 2/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9803 - loss: 0.0737 - val_accuracy: 0.9878 - val_loss: 0.0462 - learning_rate: 0.0010
Epoch 3/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9823 - loss: 0.0608 - val_accuracy: 0.9871 - val_loss: 0.0453 

In [89]:
ablation_feature_cols = behavior_feature_cols + [
    "mean_intertweet_seconds",
    "intertweet_cv",
    "posting_time_entropy",
    "active_hour_count",
    "top_hour_concentration"
]

In [90]:
ablation_feature_cols = behavior_feature_cols + [
    "mean_intertweet_seconds",
    "intertweet_cv",
    "posting_time_entropy",
    "active_hour_count",
    "top_hour_concentration"
]

print("Total features:", len(ablation_feature_cols))
print("Features:", ablation_feature_cols)

Total features: 20
Features: ['statuses_count', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'account_age_days', 'statuses_per_day', 'followers_friends_ratio', 'friends_followers_ratio', 'favorites_per_status', 'log_statuses_count', 'log_followers_count', 'log_friends_count', 'log_favourites_count', 'log_listed_count', 'mean_intertweet_seconds', 'intertweet_cv', 'posting_time_entropy', 'active_hour_count', 'top_hour_concentration']


In [88]:
# ============================================================
# EVALUATION
# ============================================================

ab_test_prob = ablation_nn.predict(
    X_test_ab_scaled,
    verbose=0
).ravel()

ab_test_pred = (ab_test_prob >= 0.5).astype(int)

ab_acc = accuracy_score(y_test_temporal, ab_test_pred)
ab_precision = precision_score(y_test_temporal, ab_test_pred)
ab_recall = recall_score(y_test_temporal, ab_test_pred)
ab_f1 = f1_score(y_test_temporal, ab_test_pred)
ab_cm = confusion_matrix(y_test_temporal, ab_test_pred)

print("\nTEMPORAL ABLATION 7 — BEHAVIOR + INTERTWEET + ACTIVITY")
print("=" * 70)
print(f"Accuracy :  {ab_acc * 100:.2f}%")
print(f"Precision:  {ab_precision * 100:.2f}%")
print(f"Recall   :  {ab_recall * 100:.2f}%")
print(f"F1 Score :  {ab_f1 * 100:.2f}%")

print("\nConfusion Matrix:")
print(ab_cm)


TEMPORAL ABLATION 7 — BEHAVIOR + INTERTWEET + ACTIVITY
Accuracy :  98.90%
Precision:  99.69%
Recall   :  98.84%
F1 Score :  99.26%

Confusion Matrix:
[[1073   10]
 [  37 3165]]


In [91]:
# ============================================================
# TEMPORAL ABLATION 8
# BEHAVIOR + INTERTWEET + POSTING-TIME
# ============================================================

ablation_feature_cols = behavior_feature_cols + [
    "mean_intertweet_seconds",
    "intertweet_cv",
    "posting_time_entropy",
    "active_hour_count",
    "top_hour_concentration"
]

print("Total features:", len(ablation_feature_cols))
print("Features:", ablation_feature_cols)


# ============================================================
# 1. PREPARE DATA
# ============================================================

X_ablation = lobo_train[ablation_feature_cols].copy()
X_ablation_test = lobo_test[ablation_feature_cols].copy()

X_train_ab, X_val_ab, y_train_ab, y_val_ab = train_test_split(
    X_ablation,
    y_lobo,
    test_size=0.20,
    random_state=42,
    stratify=y_lobo
)


# ============================================================
# 2. IMPUTATION — TRAINING DATA ONLY
# ============================================================

ab_imputer = SimpleImputer(strategy="median")

X_train_ab_imp = ab_imputer.fit_transform(X_train_ab)
X_val_ab_imp = ab_imputer.transform(X_val_ab)
X_test_ab_imp = ab_imputer.transform(X_ablation_test)


# ============================================================
# 3. SCALING — TRAINING DATA ONLY
# ============================================================

ab_scaler = StandardScaler()

X_train_ab_scaled = ab_scaler.fit_transform(X_train_ab_imp)
X_val_ab_scaled = ab_scaler.transform(X_val_ab_imp)
X_test_ab_scaled = ab_scaler.transform(X_test_ab_imp)


# ============================================================
# 4. CLASS WEIGHTS
# ============================================================

classes = np.unique(y_train_ab)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_ab
)

ab_class_weights = dict(zip(classes, class_weights_values))

print("Class weights:", ab_class_weights)


# ============================================================
# 5. NEURAL NETWORK
# ============================================================

tf.random.set_seed(42)
np.random.seed(42)

ablation_nn = Sequential([
    Input(shape=(X_train_ab_scaled.shape[1],)),

    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.25),

    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.20),

    Dense(32, activation="relu"),
    Dropout(0.10),

    Dense(1, activation="sigmoid")
])

ablation_nn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


# ============================================================
# 6. CALLBACKS
# ============================================================

early_stopping_ab = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr_ab = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)


# ============================================================
# 7. TRAIN
# ============================================================

history_ablation = ablation_nn.fit(
    X_train_ab_scaled,
    y_train_ab,
    validation_data=(X_val_ab_scaled, y_val_ab),
    epochs=50,
    batch_size=32,
    class_weight=ab_class_weights,
    callbacks=[
        early_stopping_ab,
        reduce_lr_ab
    ],
    verbose=1
)


# ============================================================
# 8. TEST PREDICTIONS
# ============================================================

ab_test_prob = ablation_nn.predict(
    X_test_ab_scaled,
    verbose=0
).ravel()

ab_test_pred = (ab_test_prob >= 0.5).astype(int)


# ============================================================
# 9. EVALUATION
# ============================================================

ab_acc = accuracy_score(
    y_test_temporal,
    ab_test_pred
)

ab_precision = precision_score(
    y_test_temporal,
    ab_test_pred
)

ab_recall = recall_score(
    y_test_temporal,
    ab_test_pred
)

ab_f1 = f1_score(
    y_test_temporal,
    ab_test_pred
)

ab_cm = confusion_matrix(
    y_test_temporal,
    ab_test_pred
)


# ============================================================
# 10. RESULTS
# ============================================================

print("\nTEMPORAL ABLATION 8 — BEHAVIOR + INTERTWEET + POSTING-TIME")
print("=" * 70)

print(f"Accuracy :  {ab_acc * 100:.2f}%")
print(f"Precision:  {ab_precision * 100:.2f}%")
print(f"Recall   :  {ab_recall * 100:.2f}%")
print(f"F1 Score :  {ab_f1 * 100:.2f}%")

print("\nConfusion Matrix:")
print(ab_cm)

Total features: 20
Features: ['statuses_count', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'account_age_days', 'statuses_per_day', 'followers_friends_ratio', 'friends_followers_ratio', 'favorites_per_status', 'log_statuses_count', 'log_followers_count', 'log_friends_count', 'log_favourites_count', 'log_listed_count', 'mean_intertweet_seconds', 'intertweet_cv', 'posting_time_entropy', 'active_hour_count', 'top_hour_concentration']
Class weights: {np.int64(0): np.float64(3.23094688221709), np.int64(1): np.float64(0.5915433403805497)}
Epoch 1/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9350 - loss: 0.1521 - val_accuracy: 0.9900 - val_loss: 0.0739 - learning_rate: 0.0010
Epoch 2/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9812 - loss: 0.0711 - val_accuracy: 0.9886 - val_loss: 0.0466 - learning_rate: 0.0010
Epoch 3/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9828 - loss: 0.0574 - val_accuracy: 0.9886 - val_loss: 0.0413 -

In [92]:
# ============================================================
# EXPERIMENT 9 — FINAL TEMPORAL IMPROVEMENT ANALYSIS
# Compare Behavioral Baseline vs Best Temporal Model
# ============================================================

baseline_accuracy = 98.44
baseline_f1 = 98.95

best_temporal_accuracy = 98.90
best_temporal_f1 = 99.26

accuracy_improvement = best_temporal_accuracy - baseline_accuracy
f1_improvement = best_temporal_f1 - baseline_f1

print("=" * 70)
print("FINAL TEMPORAL IMPROVEMENT ANALYSIS")
print("=" * 70)

print("\nBaseline Behavioral Model")
print(f"Accuracy : {baseline_accuracy:.2f}%")
print(f"F1 Score : {baseline_f1:.2f}%")

print("\nBest Temporal Model")
print("Features: Behavior + Intertweet + Activity Variability")
print(f"Accuracy : {best_temporal_accuracy:.2f}%")
print(f"F1 Score : {best_temporal_f1:.2f}%")

print("\nImprovement")
print(f"Accuracy improvement: +{accuracy_improvement:.2f} percentage points")
print(f"F1 improvement      : +{f1_improvement:.2f} percentage points")

print("\nConclusion:")
if accuracy_improvement > 0 and f1_improvement > 0:
    print("Temporal behavioral features improved both accuracy and F1.")
else:
    print("Temporal features did not improve both metrics.")

FINAL TEMPORAL IMPROVEMENT ANALYSIS

Baseline Behavioral Model
Accuracy : 98.44%
F1 Score : 98.95%

Best Temporal Model
Features: Behavior + Intertweet + Activity Variability
Accuracy : 98.90%
F1 Score : 99.26%

Improvement
Accuracy improvement: +0.46 percentage points
F1 improvement      : +0.31 percentage points

Conclusion:
Temporal behavioral features improved both accuracy and F1.


In [93]:
# ============================================================
# FINAL TEMPORAL MODEL COMPARISON
# ============================================================

temporal_results = pd.DataFrame([
    {
        "Experiment": "Behavioral Baseline",
        "Feature Set": "15 Behavioral",
        "Accuracy": 98.44,
        "Precision": 99.75,
        "Recall": 98.16,
        "F1": 98.95
    },
    {
        "Experiment": "Intertweet Only",
        "Feature Set": "2 Temporal",
        "Accuracy": 90.25,
        "Precision": 96.22,
        "Recall": 90.51,
        "F1": 93.27
    },
    {
        "Experiment": "Posting-time Only",
        "Feature Set": "3 Temporal",
        "Accuracy": 88.68,
        "Precision": 97.62,
        "Recall": 86.98,
        "F1": 91.99
    },
    {
        "Experiment": "Activity Variability Only",
        "Feature Set": "3 Temporal",
        "Accuracy": 82.38,
        "Precision": 92.47,
        "Recall": 83.20,
        "F1": 87.59
    },
    {
        "Experiment": "Behavior + Posting-time",
        "Feature Set": "18 Features",
        "Accuracy": 98.69,
        "Precision": 99.68,
        "Recall": 98.56,
        "F1": 99.12
    },
    {
        "Experiment": "Behavior + Activity",
        "Feature Set": "18 Features",
        "Accuracy": 98.76,
        "Precision": 99.72,
        "Recall": 98.63,
        "F1": 99.17
    },
    {
        "Experiment": "Behavior + Intertweet",
        "Feature Set": "17 Features",
        "Accuracy": 98.83,
        "Precision": 99.53,
        "Recall": 98.91,
        "F1": 99.22
    },
    {
        "Experiment": "Behavior + Intertweet + Activity",
        "Feature Set": "20 Features",
        "Accuracy": 98.90,
        "Precision": 99.69,
        "Recall": 98.84,
        "F1": 99.26
    },
    {
        "Experiment": "Behavior + Intertweet + Posting-time",
        "Feature Set": "20 Features",
        "Accuracy": 98.20,
        "Precision": 99.90,
        "Recall": 97.69,
        "F1": 98.78
    },
    {
        "Experiment": "Full Temporal",
        "Feature Set": "23 Features",
        "Accuracy": 98.18,
        "Precision": 99.81,
        "Recall": 97.75,
        "F1": 98.77
    }
])

temporal_results

,Experiment,Feature Set,Accuracy,Precision,Recall,F1
0,Behavioral Baseline,15 Behavioral,98.44,99.75,98.16,98.95
1,Intertweet Only,2 Temporal,90.25,96.22,90.51,93.27
2,Posting-time Only,3 Temporal,88.68,97.62,86.98,91.99
3,Activity Variability Only,3 Temporal,82.38,92.47,83.20,87.59
4,Behavior + Posting-time,18 Features,98.69,99.68,98.56,99.12
5,Behavior + Activity,18 Features,98.76,99.72,98.63,99.17
6,Behavior + Intertweet,17 Features,98.83,99.53,98.91,99.22
7,Behavior + Intertweet + Activity,20 Features,98.90,99.69,98.84,99.26
8,Behavior + Intertweet + Posting-time,20 Features,98.20,99.90,97.69,98.78
9,Full Temporal,23 Features,98.18,99.81,97.75,98.77


In [94]:
# ============================================================
# BEST TEMPORAL MODEL
# ============================================================

best_temporal = temporal_results.loc[
    temporal_results["F1"].idxmax()
]

print("BEST TEMPORAL MODEL")
print("=" * 60)

print("Experiment :", best_temporal["Experiment"])
print("Feature Set :", best_temporal["Feature Set"])
print(f"Accuracy    : {best_temporal['Accuracy']:.2f}%")
print(f"Precision   : {best_temporal['Precision']:.2f}%")
print(f"Recall      : {best_temporal['Recall']:.2f}%")
print(f"F1 Score    : {best_temporal['F1']:.2f}%")

BEST TEMPORAL MODEL
Experiment : Behavior + Intertweet + Activity
Feature Set : 20 Features
Accuracy    : 98.90%
Precision   : 99.69%
Recall      : 98.84%
F1 Score    : 99.26%


In [95]:
# ============================================================
# IMPROVEMENT OVER BASELINE
# ============================================================

baseline = temporal_results[
    temporal_results["Experiment"] == "Behavioral Baseline"
].iloc[0]

best = temporal_results.loc[
    temporal_results["F1"].idxmax()
]

print("IMPROVEMENT OVER BEHAVIORAL BASELINE")
print("=" * 60)

print(
    f"Accuracy: {baseline['Accuracy']:.2f}% → "
    f"{best['Accuracy']:.2f}% "
    f"(+{best['Accuracy'] - baseline['Accuracy']:.2f} pp)"
)

print(
    f"F1 Score: {baseline['F1']:.2f}% → "
    f"{best['F1']:.2f}% "
    f"(+{best['F1'] - baseline['F1']:.2f} pp)"
)

IMPROVEMENT OVER BEHAVIORAL BASELINE
Accuracy: 98.44% → 98.90% (+0.46 pp)
F1 Score: 98.95% → 99.26% (+0.31 pp)


## Key Findings

The temporal ablation study evaluated three groups of temporal behavioral features:
intertweet timing, posting-time behavior, and activity variability.

The strongest configuration combined the original 15 behavioral features with
two intertweet timing features (`mean_intertweet_seconds` and `intertweet_cv`)
and three activity-variability features (`daily_activity_cv`,
`night_activity_ratio`, and `weekend_activity_ratio`).

This 20-feature configuration achieved 98.90% accuracy and 99.26% F1-score
on the unseen Fake Followers evaluation, compared with 98.44% accuracy and
98.95% F1-score for the 15-feature behavioral baseline.

This corresponds to an improvement of 0.46 percentage points in accuracy and
0.31 percentage points in F1-score.

The ablation results also demonstrate that adding more temporal features does
not necessarily improve generalization. The full 23-feature configuration
performed worse than the selected 20-feature configuration. Therefore,
temporal features should be evaluated selectively rather than assuming that
more temporal information will always improve performance.

These results should be interpreted as exploratory because multiple temporal
configurations were evaluated on the same unseen-category test set.